# Descriptivo prueba técnica
En este Jupyter Notebook se consolida todo el código empleado en el desarrollo del descriptivo generado para el desarrollo de la prueba técnica.

## 1. Importar librerías y configurar variables globales

In [1]:
import sys
print(sys.prefix)
from dateutil.relativedelta import relativedelta
import pathlib
import pandas as pd
import random
import numpy as np
from sparky_bc import Sparky
from helper.helper import Helper
import getpass as gp
import os

c:\Users\seaherre\Documents\Repositorios\prueba-tecnica\venv


In [2]:
# Configuración general
dsn = 'impala-virtual-prd'
ruta_rutina = pathlib.Path().resolve()
ruta_general = pathlib.Path().resolve().parent

# Rutas
sql = ruta_rutina / 'sql'
data = ruta_rutina / 'data'
data_final =  ruta_general / 'torneo_modelos' / 'data'
ins = data / 'insumos'

# Zonas de trabajo
zona_pr = 'proceso'
zona_p = 'proceso'
zona_r = 'proceso'
indice = 'prutecge'

### Inicializar conexiones

In [3]:
hp = Helper(dsn = dsn)
sp = Sparky(username = gp.getuser(), password = os.environ.get("network_password"), dsn = dsn)

2026-09-13 07:54:29 - [WARNING] - No se encontro la carpeta "c:\Users\seaherre\Documents\Repositorios\prueba-tecnica\src\descriptivo\logs" para guardar los logs


 ___ __  __ ____   _    _        _    
|_ _|  \/  |  _ \ / \  | |      / \   
 | || |\/| | |_) / _ \ | |     / _ \  
 | || |  | |  __/ ___ \| |___ / ___ \ 
|___|_|  |_|_| /_/   \_\_____/_/   \_\
                                      
 _   _ _____ _     ____  _____ ____  
| | | | ____| |   |  _ \| ____|  _ \ 
| |_| |  _| | |   | |_) |  _| | |_) |
|  _  | |___| |___|  __/| |___|  _ < 
|_| |_|_____|_____|_|   |_____|_| \_\
                                     



2026-09-13 07:54:31 - [WARNING] - No se encontro la carpeta "c:\Users\seaherre\Documents\Repositorios\prueba-tecnica\src\descriptivo\logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



## 2. Diccionario de insumos y bases de datos

In [4]:
insumos = {
    'base_trtest': {
        'archivo': 'prueba_op_base_pivot_var_rpta_alt_enmascarado_trtest.csv',
        'descripcion': 'Base de datos que contiene la variable respuesta y las principales características de la gestión, de resultados del pago y de características de las opciones de pago habilitadas para el cliente en el mes de gestión o de evaluación de la variable respuesta.',
    },
    'base_oot': {
        'archivo': 'prueba_op_base_pivot_var_rpta_alt_enmascarado_oot.csv',
        'descripcion': 'Lista de clientes y obligaciones a calificar.'
    },
    'base_info_modelos': {
        'archivo': 'prueba_op_probabilidad_oblig_base_hist_enmascarado_completa.csv',
        'descripcion': 'Información asociada con los resultados de los modelos analíticos existentes para la cobranza de Bancolombia. Incluye: Alerta temprana (probabilidad que un cliente entre en mora), Auto cura (probabilidad de pago automático), Propensión de pago.',
    },
    'base_info_socio': {
        'archivo': 'prueba_op_master_customer_data_enmascarado_completa.csv',
        'descripcion': 'Información asociada a las características generales del cliente o demográficas de forma mensual.',
    },
    'base_info_pagos': {
        'archivo': 'prueba_op_maestra_cuotas_pagos_mes_hist_enmascarado_completa.csv',
        'descripcion': 'Información que describe el comportamiento de pagos del cliente en sus obligaciones a lo largo del tiempo.',
    }
}

## 3. Funciones auxiliares de análisis

In [5]:
### 3.1 Función: Calcular porcentaje por fecha
def calcular_porcentaje_por_fecha(df, col_fecha='fecha_var_rpta_alt', col_respuesta='var_rpta_alt'):
    """
    Agrupa datos por fecha y tipo de respuesta, calculando porcentajes.
    
    Args:
        df (pd.DataFrame): DataFrame a procesar
        col_fecha (str): Nombre de la columna de fecha (default: 'fecha_var_rpta_alt')
        col_respuesta (str): Nombre de la columna de respuesta (default: 'var_rpta_alt')
    
    Returns:
        pd.DataFrame: DataFrame con columnas fecha, respuesta, cantidad y porcentaje
    """
    resultado = df.groupby([col_fecha, col_respuesta]).size().reset_index(name='cantidad_casos')
    resultado = resultado.sort_values([col_fecha, col_respuesta], ascending=True)
    resultado['total_por_fecha'] = resultado.groupby(col_fecha)['cantidad_casos'].transform('sum')
    resultado['porcentaje'] = (resultado['cantidad_casos'] / resultado['total_por_fecha'] * 100).round(2)
    resultado = resultado.drop('total_por_fecha', axis=1)
    
    return resultado

In [6]:
### 3.2 Función auxiliar: Mostrar resultados de proporciones
def mostrar_resultados(resultado):
    """
    Muestra los resultados del análisis de proporciones de forma visual y clara
    """
    if resultado is None:
        return
    
    segmento = resultado['segmento']
    general = resultado['general']
    df_por_fecha = resultado['por_fecha']
    
    print("\n" + "="*80)
    print(f"ANÁLISIS DE PROPORCIONES - SEGMENTO: {segmento}")
    print("="*80)
    
    # Proporciones generales
    print("\n📊 PROPORCIONES GENERALES (TODA LA BASE):")
    print("-"*80)
    print(f"  Total de registros: {general['total']:,}")
    print(f"  Ceros (0): {general['ceros_cantidad']:,} casos → {general['ceros_porcentaje']}%")
    print(f"  Unos  (1): {general['unos_cantidad']:,} casos → {general['unos_porcentaje']}%")
    
    # Proporciones por fecha
    print("\n📅 PROPORCIONES POR FECHA:")
    print("-"*80)
    
    df_display = df_por_fecha.copy()
    df_display.columns = ['Fecha', 'Total', 'Ceros', 'Unos', '% Ceros', '% Unos']
    print(df_display.to_string(index=False))
    
    # Estadísticas
    print("\n" + "-"*80)
    print("📈 ESTADÍSTICAS POR FECHA:")
    print(f"  Total de fechas: {len(df_por_fecha)}")
    print(f"  Promedio % ceros: {df_por_fecha['pct_ceros'].mean():.2f}%")
    print(f"  Promedio % unos:  {df_por_fecha['pct_unos'].mean():.2f}%")
    print(f"  Máximo % ceros:   {df_por_fecha['pct_ceros'].max():.2f}%")
    print(f"  Máximo % unos:    {df_por_fecha['pct_unos'].max():.2f}%")
    print(f"  Mínimo % ceros:   {df_por_fecha['pct_ceros'].min():.2f}%")
    print(f"  Mínimo % unos:    {df_por_fecha['pct_unos'].min():.2f}%")
    
    print("="*80 + "\n")

In [7]:
### 3.3 Función: Analizar proporciones por segmento
def analizar_proporciones_segmento(
    df,
    segmento_valor,
    col_segmento='segmento',
    col_fecha='fecha_var_rpta_alt',
    col_respuesta='var_rpta_alt',
    mostrar_detalle=True
):
    """
    Calcula las proporciones de ceros y unos por fecha y general para un segmento.
    
    Args:
        df: DataFrame con los datos
        segmento_valor: Valor del segmento a analizar (ej: 'COMERCIAL')
        col_segmento: Nombre de la columna segmento (default: 'segmento')
        col_fecha: Nombre de la columna fecha (default: 'fecha_var_rpta_alt')
        col_respuesta: Nombre de la columna con 0/1 (default: 'var_rpta_alt')
        mostrar_detalle: Si mostrar tabla detallada (default: True)
    
    Returns:
        Diccionario con 'por_fecha', 'general' y 'segmento'
    """
    
    try:
        df_segmento = df[df[col_segmento] == segmento_valor].copy()
        
        if len(df_segmento) == 0:
            print(f"❌ No hay registros para el segmento: {segmento_valor}")
            return None
        
        # Proporciones generales
        total_general = len(df_segmento)
        ceros_general = (df_segmento[col_respuesta] == 0).sum()
        unos_general = (df_segmento[col_respuesta] == 1).sum()
        
        proporciones_general = {
            'total': total_general,
            'ceros_cantidad': ceros_general,
            'ceros_porcentaje': round((ceros_general / total_general * 100), 2),
            'unos_cantidad': unos_general,
            'unos_porcentaje': round((unos_general / total_general * 100), 2)
        }
        
        # Proporciones por fecha
        df_por_fecha = df_segmento.groupby(col_fecha).apply(
            lambda x: pd.Series({
                'total': len(x),
                'ceros': (x[col_respuesta] == 0).sum(),
                'unos': (x[col_respuesta] == 1).sum()
            })
        ).reset_index()
        
        df_por_fecha['pct_ceros'] = (df_por_fecha['ceros'] / df_por_fecha['total'] * 100).round(2)
        df_por_fecha['pct_unos'] = (df_por_fecha['unos'] / df_por_fecha['total'] * 100).round(2)
        df_por_fecha = df_por_fecha.sort_values(col_fecha).reset_index(drop=True)
        
        resultado = {
            'por_fecha': df_por_fecha,
            'general': proporciones_general,
            'segmento': segmento_valor
        }
        
        if mostrar_detalle:
            mostrar_resultados(resultado)
        
        return resultado
    
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return None

In [8]:
### 3.4 Función: Crear resumen de todos los segmentos
def crear_resumen_todos_segmentos(df, col_segmento='segmento'):
    """
    Crea un resumen de proporciones generales para todos los segmentos.
    
    Args:
        df: DataFrame con los datos
        col_segmento: Nombre de la columna segmento
    
    Returns:
        pd.DataFrame: DataFrame con resumen de todos los segmentos
    """
    segmentos = df[col_segmento].unique()
    resumen = []
    
    for segmento in segmentos:
        resultado = analizar_proporciones_segmento(
            df,
            segmento_valor=segmento,
            mostrar_detalle=False
        )
        if resultado:
            gen = resultado['general']
            resumen.append({
                'Segmento': segmento,
                'Total': gen['total'],
                'Ceros': gen['ceros_cantidad'],
                '% Ceros': gen['ceros_porcentaje'],
                'Unos': gen['unos_cantidad'],
                '% Unos': gen['unos_porcentaje']
            })
    
    df_resumen = pd.DataFrame(resumen)
    return df_resumen

In [9]:
### 3.5 Función auxiliar: Mostrar mapeo de clientes
def mostrar_mapeo_clientes(resultado):
    """
    Muestra el mapeo de clientes recurrentes de forma visual
    """
    if resultado is None:
        return
    
    segmento = resultado['segmento']
    stats = resultado['estadisticas']
    df_recurrentes = resultado['clientes_recurrentes']
    frecuencia = resultado['frecuencia_fechas']
    
    print("\n" + "="*100)
    print(f"MAPEO DE CLIENTES RECURRENTES - SEGMENTO: {segmento}")
    print("="*100)
    
    # Estadísticas generales
    print("\n📊 ESTADÍSTICAS GENERALES:")
    print("-"*100)
    print(f"  Total clientes únicos: {stats['total_clientes_unicos']:,}")
    print(f"  Clientes recurrentes: {stats['total_clientes_recurrentes']:,} ({stats['porcentaje_recurrentes']}%)")
    print(f"  Total registros: {stats['total_registros']:,}")
    print(f"  Fechas diferentes: {stats['total_fechas']}")
    print(f"  Promedio fechas por cliente: {stats['promedio_fechas_por_cliente']}")
    print(f"  Máximas fechas de un cliente: {stats['max_fechas_cliente']}")
    print(f"  Promedio registros por cliente: {stats['promedio_registros_por_cliente']}")
    
    # Distribución
    print("\n📈 DISTRIBUCIÓN DE FRECUENCIA:")
    print("-"*100)
    for fechas, cantidad in frecuencia.items():
        porcentaje = round((cantidad / stats['total_clientes_unicos'] * 100), 2)
        barra = "█" * int(cantidad / 5) if cantidad >= 5 else "█"
        print(f"  Clientes con {fechas} fecha(s): {cantidad:5} ({porcentaje:5.2f}%) {barra}")
    
    # Top clientes
    print("\n🔝 TOP 10 CLIENTES MÁS RECURRENTES:")
    print("-"*100)
    df_top = df_recurrentes.head(10).copy()
    
    for idx, row in df_top.iterrows():
        print(f"\n  NIT: {row['nit_enmascarado']}")
        print(f"    Fechas visitadas: {row['cantidad_fechas']}")
        print(f"    Registros totales: {row['cantidad_registros']}")
        print(f"    Promedio por fecha: {row['promedio_por_fecha']}")
        print(f"    Fechas: {row['fechas']}")
    
    print("\n" + "="*100 + "\n")

In [10]:
### 3.6 Función: Mapear clientes recurrentes
def mapear_clientes_recurrentes(
    df,
    col_nit='nit_enmascarado',
    col_fecha='fecha_var_rpta_alt',
    col_segmento='segmento',
    segmento_filtro=None,
    min_fechas=2,
    mostrar_detalle=True
):
    """
    Mapea clientes que se repiten en diferentes fechas.
    
    Args:
        df: DataFrame con los datos
        col_nit: Nombre columna NIT enmascarado
        col_fecha: Nombre columna fecha
        col_segmento: Nombre columna segmento
        segmento_filtro: Filtrar por segmento específico (None = todos)
        min_fechas: Mínimo de fechas para considerar recurrente
        mostrar_detalle: Si mostrar tabla detallada
    
    Returns:
        Diccionario con clientes_recurrentes, frecuencia_fechas, detalle_cruce y estadísticas
    """

    try:
        df_trabajo = df.copy()
        if segmento_filtro is not None:
            df_trabajo = df_trabajo[df_trabajo[col_segmento] == segmento_filtro]
            if len(df_trabajo) == 0:
                print(f"❌ No hay registros para el segmento: {segmento_filtro}")
                return None
        
        # Contar fechas por cliente
        cliente_fechas = df_trabajo.groupby(col_nit)[col_fecha].nunique().reset_index()
        cliente_fechas.columns = [col_nit, 'cantidad_fechas']
        cliente_fechas = cliente_fechas.sort_values('cantidad_fechas', ascending=False)
        
        # Identificar recurrentes
        clientes_recurrentes = cliente_fechas[cliente_fechas['cantidad_fechas'] >= min_fechas].copy()
        
        # Listar fechas por cliente recurrente
        fechas_por_cliente = []
        for nit in clientes_recurrentes[col_nit]:
            fechas = df_trabajo[df_trabajo[col_nit] == nit][col_fecha].unique()
            fechas_ordenadas = sorted(fechas)
            cantidad_registros = len(df_trabajo[df_trabajo[col_nit] == nit])
            
            fechas_por_cliente.append({
                col_nit: nit,
                'cantidad_fechas': len(fechas_ordenadas),
                'fechas': ', '.join([str(f) for f in fechas_ordenadas]),
                'cantidad_registros': cantidad_registros,
                'promedio_por_fecha': round(cantidad_registros / len(fechas_ordenadas), 2)
            })
        
        df_recurrentes_detalle = pd.DataFrame(fechas_por_cliente)
        
        # Detalle completo: cliente x fecha
        detalle_cruce = df_trabajo.groupby([col_nit, col_fecha]).size().reset_index(name='cantidad')
        detalle_cruce = detalle_cruce.sort_values([col_nit, col_fecha])
        
        # Estadísticas
        total_clientes = df_trabajo[col_nit].nunique()
        total_registros = len(df_trabajo)
        
        estadisticas = {
            'total_clientes_unicos': total_clientes,
            'total_clientes_recurrentes': len(clientes_recurrentes),
            'porcentaje_recurrentes': round((len(clientes_recurrentes) / total_clientes * 100), 2),
            'total_registros': total_registros,
            'total_fechas': df_trabajo[col_fecha].nunique(),
            'promedio_fechas_por_cliente': round(cliente_fechas['cantidad_fechas'].mean(), 2),
            'max_fechas_cliente': cliente_fechas['cantidad_fechas'].max(),
            'promedio_registros_por_cliente': round(total_registros / total_clientes, 2)
        }
        
        # Distribución de frecuencia
        frecuencia_fechas = cliente_fechas['cantidad_fechas'].value_counts().sort_index()
        
        resultado = {
            'clientes_recurrentes': df_recurrentes_detalle,
            'frecuencia_fechas': frecuencia_fechas,
            'detalle_cruce': detalle_cruce,
            'cliente_fechas_todas': cliente_fechas,
            'estadisticas': estadisticas,
            'segmento': segmento_filtro if segmento_filtro else 'TODOS'
        }
        
        if mostrar_detalle:
            mostrar_mapeo_clientes(resultado)
        
        return resultado
    
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return None

In [11]:
### 3.7 Función auxiliar: Mostrar resultados de proporciones por producto
def mostrar_resultados_producto(resultado):
    """
    Muestra los resultados del análisis de proporciones por producto de forma visual y clara
    """
    if resultado is None:
        return
    
    producto = resultado['producto']
    general = resultado['general']
    df_por_fecha = resultado['por_fecha']
    
    print("\n" + "="*80)
    print(f"ANÁLISIS DE PROPORCIONES - PRODUCTO: {producto}")
    print("="*80)
    
    # Proporciones generales
    print("\n📊 PROPORCIONES GENERALES (TODA LA BASE):")
    print("-"*80)
    print(f"  Total de registros: {general['total']:,}")
    print(f"  Ceros (0): {general['ceros_cantidad']:,} casos → {general['ceros_porcentaje']}%")
    print(f"  Unos  (1): {general['unos_cantidad']:,} casos → {general['unos_porcentaje']}%")
    
    # Proporciones por fecha
    print("\n📅 PROPORCIONES POR FECHA:")
    print("-"*80)
    
    df_display = df_por_fecha.copy()
    df_display.columns = ['Fecha', 'Total', 'Ceros', 'Unos', '% Ceros', '% Unos']
    print(df_display.to_string(index=False))
    
    # Estadísticas
    print("\n" + "-"*80)
    print("📈 ESTADÍSTICAS POR FECHA:")
    print(f"  Total de fechas: {len(df_por_fecha)}")
    print(f"  Promedio % ceros: {df_por_fecha['pct_ceros'].mean():.2f}%")
    print(f"  Promedio % unos:  {df_por_fecha['pct_unos'].mean():.2f}%")
    print(f"  Máximo % ceros:   {df_por_fecha['pct_ceros'].max():.2f}%")
    print(f"  Máximo % unos:    {df_por_fecha['pct_unos'].max():.2f}%")
    print(f"  Mínimo % ceros:   {df_por_fecha['pct_ceros'].min():.2f}%")
    print(f"  Mínimo % unos:    {df_por_fecha['pct_unos'].min():.2f}%")
    
    print("="*80 + "\n")

In [12]:
### 3.8 Función: Analizar proporciones por producto
def analizar_proporciones_producto(
    df,
    producto_valor,
    col_producto='producto',
    col_fecha='fecha_var_rpta_alt',
    col_respuesta='var_rpta_alt',
    mostrar_detalle=True
):
    """
    Calcula las proporciones de ceros y unos por fecha y general para un producto.
    
    Args:
        df: DataFrame con los datos
        producto_valor: Valor del producto a analizar
        col_producto: Nombre de la columna producto (default: 'producto')
        col_fecha: Nombre de la columna fecha (default: 'fecha_var_rpta_alt')
        col_respuesta: Nombre de la columna con 0/1 (default: 'var_rpta_alt')
        mostrar_detalle: Si mostrar tabla detallada (default: True)
    
    Returns:
        Diccionario con 'por_fecha', 'general' y 'producto'
    """
    
    try:
        df_producto = df[df[col_producto] == producto_valor].copy()
        
        if len(df_producto) == 0:
            print(f"❌ No hay registros para el producto: {producto_valor}")
            return None
        
        # Proporciones generales
        total_general = len(df_producto)
        ceros_general = (df_producto[col_respuesta] == 0).sum()
        unos_general = (df_producto[col_respuesta] == 1).sum()
        
        proporciones_general = {
            'total': total_general,
            'ceros_cantidad': ceros_general,
            'ceros_porcentaje': round((ceros_general / total_general * 100), 2),
            'unos_cantidad': unos_general,
            'unos_porcentaje': round((unos_general / total_general * 100), 2)
        }
        
        # Proporciones por fecha
        df_por_fecha = df_producto.groupby(col_fecha).apply(
            lambda x: pd.Series({
                'total': len(x),
                'ceros': (x[col_respuesta] == 0).sum(),
                'unos': (x[col_respuesta] == 1).sum()
            })
        ).reset_index()
        
        df_por_fecha['pct_ceros'] = (df_por_fecha['ceros'] / df_por_fecha['total'] * 100).round(2)
        df_por_fecha['pct_unos'] = (df_por_fecha['unos'] / df_por_fecha['total'] * 100).round(2)
        df_por_fecha = df_por_fecha.sort_values(col_fecha).reset_index(drop=True)
        
        resultado = {
            'por_fecha': df_por_fecha,
            'general': proporciones_general,
            'producto': producto_valor
        }
        
        if mostrar_detalle:
            mostrar_resultados_producto(resultado)
        
        return resultado
    
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return None

In [13]:
### 3.9 Función: Crear resumen de todos los productos
def crear_resumen_todos_productos(df, col_producto='producto'):
    """
    Crea un resumen de proporciones generales para todos los productos.
    
    Args:
        df: DataFrame con los datos
        col_producto: Nombre de la columna producto
    
    Returns:
        pd.DataFrame: DataFrame con resumen de todos los productos
    """
    productos = df[col_producto].unique()
    resumen = []
    
    for producto in productos:
        resultado = analizar_proporciones_producto(
            df,
            producto_valor=producto,
            mostrar_detalle=False
        )
        if resultado:
            gen = resultado['general']
            resumen.append({
                'Producto': producto,
                'Total': gen['total'],
                'Ceros': gen['ceros_cantidad'],
                '% Ceros': gen['ceros_porcentaje'],
                'Unos': gen['unos_cantidad'],
                '% Unos': gen['unos_porcentaje']
            })
    
    df_resumen = pd.DataFrame(resumen)
    return df_resumen

In [14]:
### 3.10 Función: calculo porcentaje de nulos de las columnas
def calcular_porcentaje_nulos(df, columnas=None, umbral=60.0):
    """
    Calcula el porcentaje de valores nulos (NaN + None) por columna.

    Args:
        df: DataFrame a analizar
        columnas: lista de columnas a evaluar (None = todas las columnas de df)
        umbral: porcentaje a partir del cual se considera la columna para descarte

    Returns:
        dict con:
            'resumen': DataFrame con columna, cantidad_nulos, porcentaje_nulos
            'columnas_descartar': lista de columnas con porcentaje_nulos > umbral
    """
    cols = columnas if columnas is not None else df.columns.tolist()
    total_registros = len(df)

    nulos = df[cols].isnull().sum()
    porcentaje = (nulos / total_registros * 100).round(2)

    resumen = pd.DataFrame({
        'columna': cols,
        'cantidad_nulos': nulos.values,
        'porcentaje_nulos': porcentaje.values
    }).sort_values('porcentaje_nulos', ascending=False).reset_index(drop=True)

    columnas_descartar = resumen[resumen['porcentaje_nulos'] > umbral]['columna'].tolist()

    return {
        'resumen': resumen,
        'columnas_descartar': columnas_descartar
    }

In [15]:
### 3.11 Función: calculo concurrencia en valores en columnas categoricas y numericas
def calcular_concurrencia_valores(df, columnas=None, umbral=90.0, solo_categoricas=True,
                                   incluir_enteros=False, max_valores_unicos_enteros=50):
    """
    Calcula, para cada columna, la distribución de valores únicos y su
    porcentaje de concurrencia sobre el total de datos no nulos.

    Args:
        df: DataFrame a analizar
        columnas: lista de columnas a evaluar (None = todas las columnas de df)
        umbral: porcentaje del valor más frecuente a partir del cual se
                considera la columna cuasi-constante (para descarte)
        solo_categoricas: si True, filtra automáticamente columnas tipo
                           object/category/bool (ignora numéricas continuas)
        incluir_enteros: si True, además de las categóricas incluye columnas
                          enteras (int nativo, Int64 nullable, o float con
                          valores enteros y NaN) que se comporten como
                          categóricas
        max_valores_unicos_enteros: umbral de cardinalidad para considerar
                          una columna entera como categórica (evita tratar
                          IDs o conteos continuos como categorías)

    Returns:
        dict con:
            'detalle': dict {columna: DataFrame con valor, frecuencia, porcentaje}
            'resumen': DataFrame con columna, valor_dominante, porcentaje_dominante, n_valores_unicos
            'columnas_descartar': columnas con porcentaje_dominante > umbral
    """
    cols = columnas if columnas is not None else df.columns.tolist()

    def _es_entera_categorica(col):
        serie = df[col].dropna()
        if len(serie) == 0:
            return False

        # int nativo o Int64 nullable
        if pd.api.types.is_integer_dtype(df[col]):
            return serie.nunique() <= max_valores_unicos_enteros

        # float que en realidad contiene enteros (con NaN mezclados)
        if pd.api.types.is_float_dtype(df[col]):
            if not (serie % 1 == 0).all():
                return False
            return serie.nunique() <= max_valores_unicos_enteros

        return False

    if solo_categoricas:
        cols_categoricas = [
            c for c in cols
            if df[c].dtype == 'object' or str(df[c].dtype) == 'category' or df[c].dtype == 'bool'
        ]

        if incluir_enteros:
            cols_enteras = [c for c in cols if _es_entera_categorica(c)]
            cols = cols_categoricas + [c for c in cols_enteras if c not in cols_categoricas]
        else:
            cols = cols_categoricas

    detalle = {}
    resumen = []

    for col in cols:
        serie = df[col].dropna()
        total = len(serie)

        if total == 0:
            resumen.append({
                'columna': col,
                'valor_dominante': None,
                'porcentaje_dominante': 0.0,
                'n_valores_unicos': 0
            })
            detalle[col] = pd.DataFrame(columns=['valor', 'frecuencia', 'porcentaje'])
            continue

        conteo = serie.value_counts()
        df_col = pd.DataFrame({
            'valor': conteo.index,
            'frecuencia': conteo.values,
            'porcentaje': (conteo.values / total * 100).round(2)
        })
        detalle[col] = df_col

        resumen.append({
            'columna': col,
            'valor_dominante': df_col.iloc[0]['valor'],
            'porcentaje_dominante': df_col.iloc[0]['porcentaje'],
            'n_valores_unicos': serie.nunique()
        })

    resumen = pd.DataFrame(resumen).sort_values('porcentaje_dominante', ascending=False).reset_index(drop=True)
    columnas_descartar = resumen[resumen['porcentaje_dominante'] > umbral]['columna'].tolist()

    return {
        'detalle': detalle,
        'resumen': resumen,
        'columnas_descartar': columnas_descartar
    }

In [16]:
### 3.12 Función: identificación de variables categóricas y sus valores únicos
def identificar_variables_categoricas(df, columnas=None, incluir_enteros_baja_cardinalidad=False,
                                       max_valores_unicos_enteros=50):
    """
    Identifica las variables categóricas de un DataFrame y sus valores únicos.

    Args:
        df: DataFrame a analizar
        columnas: lista de columnas a evaluar (None = todas las columnas de df)
        incluir_enteros_baja_cardinalidad: si True, incluye columnas enteras
                          (o float con enteros y NaN) con pocos valores únicos
                          como si fueran categóricas
        max_valores_unicos_enteros: umbral de cardinalidad para considerar
                          una columna entera como categórica

    Returns:
        dict con:
            'variables_categoricas': lista de nombres de columnas categóricas
            'valores_unicos': dict {columna: lista ordenada de valores únicos no nulos}
            'resumen': DataFrame con columna, tipo_dato, n_valores_unicos, tiene_nulos
    """
    cols = columnas if columnas is not None else df.columns.tolist()

    def _es_entera_baja_cardinalidad(col):
        serie = df[col].dropna()
        if len(serie) == 0:
            return False
        if pd.api.types.is_integer_dtype(df[col]):
            return serie.nunique() <= max_valores_unicos_enteros
        if pd.api.types.is_float_dtype(df[col]):
            if not (serie % 1 == 0).all():
                return False
            return serie.nunique() <= max_valores_unicos_enteros
        return False

    variables_categoricas = [
        c for c in cols
        if df[c].dtype == 'object' or str(df[c].dtype) == 'category' or df[c].dtype == 'bool'
    ]

    if incluir_enteros_baja_cardinalidad:
        cols_enteras = [c for c in cols if _es_entera_baja_cardinalidad(c)]
        variables_categoricas += [c for c in cols_enteras if c not in variables_categoricas]

    valores_unicos = {}
    resumen = []

    for col in variables_categoricas:
        serie = df[col].dropna()
        try:
            valores = sorted(serie.unique().tolist())
        except TypeError:
            valores = serie.unique().tolist()

        valores_unicos[col] = valores
        resumen.append({
            'columna': col,
            'tipo_dato': str(df[col].dtype),
            'n_valores_unicos': len(valores),
            'tiene_nulos': df[col].isnull().any()
        })

    resumen = pd.DataFrame(resumen).sort_values('n_valores_unicos').reset_index(drop=True)

    return {
        'variables_categoricas': variables_categoricas,
        'valores_unicos': valores_unicos,
        'resumen': resumen
    }

In [17]:
def crear_uso_detalle(row):
    """
    Define la columna uso_detalle basada en las condiciones:
    - Si uso == "OOT" → "val"
    - Si uso == "TRAINTEST":
        - Si fecha_analisis < 202311 → "train"
        - Si fecha_analisis == 202311 → "test"
    """
    if row['uso'] == 'OOT':
        return 'val'
    elif row['uso'] == 'TRAINTEST':
        if row['fecha_analisis'] < 202311:
            return 'train'
        elif row['fecha_analisis'] == 202311:
            return 'test'
    return None

In [18]:
def crear_ajustar_uso_detalle(nits_en_ambas):
    """
    Factory function que crea la función ajustar_uso_detalle con nits_en_ambas como parámetro.
    Si el NIT aparece en ambas categorías (train y test) Y el registro está en 'test',
    marca como 'excluido_test'
    """
    def ajustar_uso_detalle(row):
        if row['nit_enmascarado'] in nits_en_ambas and row['uso_detalle'] == 'test':
            return 'excluido_test'
        return row['uso_detalle']
    return ajustar_uso_detalle

In [19]:
def limpiar_datos_numericos(df, columnas=None, reemplazar_inf_por=0, reemplazar_nan_por=0, verbose=True):
    """
    Limpia datos numéricos: reemplaza infinitos y NaNs.
    
    Args:
        df: DataFrame a limpiar (se modifica IN-PLACE)
        columnas: lista de columnas a limpiar (None = todas las numéricas)
        reemplazar_inf_por: valor para reemplazar infinitos (default: 0)
        reemplazar_nan_por: valor para reemplazar NaNs (default: 0)
        verbose: si mostrar reporte detallado
    
    Returns:
        DataFrame limpiado + dict con resumen de cambios
    """
    
    df_trabajo = df.copy()
    
    # Identificar columnas numéricas si no se especifican
    if columnas is None:
        columnas = df_trabajo.select_dtypes(include=[np.number]).columns.tolist()
    
    if verbose:
        print("="*80)
        print("REPORTE DE LIMPIEZA DE DATOS NUMÉRICOS")
        print("="*80)
        print(f"\nColumnas numéricas a procesar: {len(columnas)}")
        print(f"Columnas: {columnas}\n")
    
    resumen = {
        'total_infinitos_antes': 0,
        'total_nans_antes': 0,
        'infinitos_por_columna': {},
        'nans_por_columna': {},
        'cambios_realizados': {}
    }
    
    # PASO 1: Detectar ANTES
    for col in columnas:
        inf_count = np.isinf(df_trabajo[col]).sum()
        nan_count = df_trabajo[col].isnull().sum()
        
        resumen['total_infinitos_antes'] += inf_count
        resumen['total_nans_antes'] += nan_count
        
        if inf_count > 0:
            resumen['infinitos_por_columna'][col] = inf_count
        if nan_count > 0:
            resumen['nans_por_columna'][col] = nan_count
    
    # Mostrar estado ANTES
    if verbose:
        print("📊 ESTADO ANTES DE LA LIMPIEZA:")
        print("-"*80)
        print(f"  Total infinitos: {resumen['total_infinitos_antes']}")
        print(f"  Total NaNs: {resumen['total_nans_antes']}")
        
        if resumen['infinitos_por_columna']:
            print(f"\n  Infinitos por columna:")
            for col, cnt in sorted(resumen['infinitos_por_columna'].items(), key=lambda x: x[1], reverse=True):
                print(f"    {col}: {cnt}")
        
        if resumen['nans_por_columna']:
            print(f"\n  NaNs por columna:")
            for col, cnt in sorted(resumen['nans_por_columna'].items(), key=lambda x: x[1], reverse=True):
                print(f"    {col}: {cnt}")
    
    # PASO 2: LIMPIAR
    for col in columnas:
        # Reemplazar infinitos
        mask_inf = np.isinf(df_trabajo[col])
        if mask_inf.any():
            df_trabajo.loc[mask_inf, col] = reemplazar_inf_por
            resumen['cambios_realizados'][f"{col}_infinitos"] = mask_inf.sum()
        
        # Reemplazar NaNs
        mask_nan = df_trabajo[col].isnull()
        if mask_nan.any():
            df_trabajo.loc[mask_nan, col] = reemplazar_nan_por
            resumen['cambios_realizados'][f"{col}_nans"] = mask_nan.sum()
    
    # PASO 3: Verificar DESPUÉS
    total_inf_despues = 0
    total_nan_despues = 0
    for col in columnas:
        total_inf_despues += np.isinf(df_trabajo[col]).sum()
        total_nan_despues += df_trabajo[col].isnull().sum()
    
    if verbose:
        print("\n✅ ESTADO DESPUÉS DE LA LIMPIEZA:")
        print("-"*80)
        print(f"  Total infinitos: {total_inf_despues}")
        print(f"  Total NaNs: {total_nan_despues}")
        print(f"\n  Cambios realizados:")
        for cambio, cantidad in resumen['cambios_realizados'].items():
            print(f"    {cambio}: {cantidad}")
        
        print("\n" + "="*80)
    
    return df_trabajo, resumen

## 4. Cargar y explorar datos

### 4.1 Exploratorio inicial de la base de entrenamiento, pruebas y variable respuesta

In [20]:
base_trtes = pd.read_csv(ins / insumos['base_trtest']['archivo'])

In [21]:
print(f"Base TRTEST: {base_trtes.shape[0]:,} registros, {base_trtes.shape[1]} columnas\n")
display(base_trtes.head(10))

Base TRTEST: 568,251 registros, 49 columnas



,nit_enmascarado,num_oblig_orig_enmascarado,num_oblig_enmascarado,fecha_var_rpta_alt,var_rpta_alt,tipo_var_rpta_alt,banca,segmento,producto,producto_cons,...,porc_pago_cuota,pago_mes,porc_pago_mes,pagos_tanque,marca_debito_mora,alternativa_aplicada_agr,marca_agrupada_rgo,marca_pago,marca_alternativa,marca_alternativa_orig
0,630611,219718,863073,202308,1,a_uno_tipo_1,Banca Personas,Personal,TARJETA DE CREDITO,Tarjeta de Credito,...,0.095438,0.00,0.000000,Sin pago,NO,CONSOLIDACION,REESTRUCTURACIÓN,Sin pago,Acepta Alternativa,Acepta Alternativa
1,59412,789567,290775,202312,1,a_uno_tipo_1,Banca Personas,Personal,LIBRE INVERSION,Libre Inversion,...,0.000000,0.00,NaN,Sin pago,NO,PRORROGA,MANTENIMIENTO,Sin pago,Acepta Alternativa,Acepta Alternativa
2,277595,1045909,34433,202312,1,b_uno_tipo_2,Banca Personas,Personal,LIBRE INVERSION,Libre Inversion,...,0.000000,0.00,NaN,Sin pago,NO,NaN,NaN,Sin pago,Acepta Alternativa,Acepta Alternativa
3,26897,585786,494556,202311,1,a_uno_tipo_1,Banca Personas,Personal,ROTATIVOS,Rotativos,...,0.042117,4090.00,0.015324,Con pago,NO,PRORROGA,MANTENIMIENTO,Pago parcial,Acepta Alternativa,Acepta Alternativa
4,24588,1061389,18953,202311,1,b_uno_tipo_2,Banca Personas,Personal plus,ROTATIVOS,Rotativos,...,0.000000,0.00,0.000000,Sin pago,NO,CONSOLIDACION,REESTRUCTURACIÓN,Sin pago,Acepta Alternativa,Acepta Alternativa
5,431682,820319,260023,202309,1,b_uno_tipo_2,Independientes,Micropyme,CARTERA ORDINARIA,Cartera Ordinaria,...,0.000550,1464792.57,0.145901,Con pago,NO,NaN,NaN,Pago Total,Acepta Alternativa,Acepta Alternativa
6,353850,842237,238105,202311,0,e_cero_tipo_2,Independientes,Micropyme,CARTERA ORDINARIA,Cartera Ordinaria,...,0.000000,0.00,NaN,Sin pago,NO,NaN,NaN,Sin pago,N.A,N.A
7,377361,895357,184985,202310,0,e_cero_tipo_2,Banca Personas,Personal,LIBRE INVERSION,Libre Inversion,...,0.000000,0.00,NaN,Sin pago,NO,NaN,NaN,Sin pago,N.A,N.A
8,356666,295727,787370,202310,0,e_cero_tipo_2,Banca Personas,Personal,TARJETA DE CREDITO,Tarjeta de Credito,...,0.001461,0.00,NaN,Sin pago,NO,NaN,NaN,Sin pago,N.A,N.A
9,451678,463512,616830,202308,1,a_uno_tipo_1,Banca Personas,Personal,ROTATIVOS,Rotativos,...,0.000000,0.00,0.000000,Sin pago,NO,CONSOLIDACION,CONSOLIDACION,Sin pago,Acepta Alternativa,Acepta Alternativa


In [22]:
base_trtes.columns

Index(['nit_enmascarado', 'num_oblig_orig_enmascarado',
       'num_oblig_enmascarado', 'fecha_var_rpta_alt', 'var_rpta_alt',
       'tipo_var_rpta_alt', 'banca', 'segmento', 'producto', 'producto_cons',
       'aplicativo', 'min_mora', 'max_mora', 'dias_mora_fin', 'rango_mora',
       'vlr_obligacion', 'vlr_vencido', 'saldo_capital', 'endeudamiento',
       'desc_alternativa1', 'desc_alternativa2', 'desc_alternativa3',
       'cant_alter_posibles', 'alter_posible1_2', 'alter_posible2_2',
       'alter_posible3_2', 'cant_gestiones', 'cant_gestiones_binario', 'rpc',
       'promesas_cumplidas', 'cant_promesas_cumplidas_binario', 'cant_acuerdo',
       'cant_acuerdo_binario', 'descripcion_ranking_mejor_ult',
       'descripcion_ranking_post_ult', 'marca_alt_rank', 'marca_alt_apli',
       'valor_cuota_mes', 'pago_cuota', 'porc_pago_cuota', 'pago_mes',
       'porc_pago_mes', 'pagos_tanque', 'marca_debito_mora',
       'alternativa_aplicada_agr', 'marca_agrupada_rgo', 'marca_pago',
      

In [23]:
dup_count = base_trtes.groupby(['nit_enmascarado',
                                # 'num_oblig_orig_enmascarado', 
                                'num_oblig_enmascarado', 
                                'fecha_var_rpta_alt']).size().reset_index(name='count')
duplicados = dup_count[dup_count['count'] > 1].sort_values('count', ascending=False)
duplicados

,nit_enmascarado,num_oblig_enmascarado,fecha_var_rpta_alt,count
332308,354954,929257,202309,3
1969,1899,658165,202308,2
403003,435560,852116,202308,2
375200,404109,837362,202308,2
379084,408533,1057978,202310,2
...,...,...,...,...
237747,252402,891609,202308,2
239461,253966,668565,202311,2
239464,253966,967824,202311,2
244311,258340,908525,202310,2


In [24]:
base_trtes[(base_trtes["nit_enmascarado"]==354954) & (base_trtes["num_oblig_enmascarado"]==929257) & (base_trtes["fecha_var_rpta_alt"]==202309)]

,nit_enmascarado,num_oblig_orig_enmascarado,num_oblig_enmascarado,fecha_var_rpta_alt,var_rpta_alt,tipo_var_rpta_alt,banca,segmento,producto,producto_cons,...,porc_pago_cuota,pago_mes,porc_pago_mes,pagos_tanque,marca_debito_mora,alternativa_aplicada_agr,marca_agrupada_rgo,marca_pago,marca_alternativa,marca_alternativa_orig
523306,354954,158854,929257,202309,0,e_cero_tipo_2,Banca Personas,Personal,TARJETA DE CREDITO,Tarjeta de Credito,...,0.0,0.0,NaN,Sin pago,NO,NaN,NaN,Sin pago,N.A,N.A
560298,354954,184019,929257,202309,0,e_cero_tipo_2,Banca Personas,Personal,TARJETA DE CREDITO,Tarjeta de Credito,...,0.0,0.0,NaN,Sin pago,NO,NaN,NaN,Sin pago,N.A,N.A
560303,354954,159202,929257,202309,0,e_cero_tipo_2,Banca Personas,Personal,TARJETA DE CREDITO,Tarjeta de Credito,...,0.0,0.0,NaN,Sin pago,NO,NaN,NaN,Sin pago,N.A,N.A


In [25]:
base_trtes['fecha_referencia_1'] = pd.to_datetime(base_trtes['fecha_var_rpta_alt'].astype(str) + '01', format='%Y%m%d')
base_trtes['fecha_referencia_2'] = base_trtes['fecha_referencia_1'].apply(lambda x: x - relativedelta(months=1))
base_trtes['fecha_analisis'] = base_trtes['fecha_referencia_2'].dt.strftime('%Y%m').astype(int)
base_trtes = base_trtes.drop(['fecha_referencia_1', 'fecha_referencia_2'], axis=1)

**Nota:** Es necesario validar con cual de los dos campos de numero de obligación "num_oblig_orig_enmascarado" o "num_oblig_enmascarado" debe realizarse el cruce, teniendo en cuenta que estos dos numero pueden diferir por condiciones específicas derivadas de cada tipo de producto (ejemplo tarjetas de credito amparadas y amparadoras).

### 4.1.1 Distribución de la variable respuesta por fecha

In [26]:
# Calcular proporción de variable respuesta por fecha
df_prop_fecha = calcular_porcentaje_por_fecha(base_trtes)
print("\n📊 Proporción de variable respuesta por fecha:")
print(df_prop_fecha.to_string(index=False))


📊 Proporción de variable respuesta por fecha:
 fecha_var_rpta_alt  var_rpta_alt  cantidad_casos  porcentaje
             202308             0           58590       51.61
             202308             1           54941       48.39
             202309             0           61357       50.63
             202309             1           59828       49.37
             202310             0           62236       53.69
             202310             1           53687       46.31
             202311             0           59875       51.11
             202311             1           57271       48.89
             202312             0           53425       53.18
             202312             1           47041       46.82


### 4.1.2 Distribución de casos por segmento

In [27]:
conteos = base_trtes['segmento'].value_counts()
total = conteos.sum()

print("\n" + "="*70)
print("DISTRIBUCIÓN DE CASOS POR SEGMENTO")
print("="*70)

for segmento, cantidad in conteos.items():
    porcentaje = (cantidad / total * 100)
    barra = "█" * int(porcentaje / 2)
    print(f"{segmento:20} {cantidad:8,} ({porcentaje:5.2f}%) {barra}")

print("="*70)
print(f"{'TOTAL':20} {total:8,} (100.00%)")
print("="*70)


DISTRIBUCIÓN DE CASOS POR SEGMENTO
Personal              361,771 (63.66%) ███████████████████████████████
Personal plus         110,014 (19.36%) █████████
Micropyme              56,746 ( 9.99%) ████
Pymes                  18,798 ( 3.31%) █
Social                 18,007 ( 3.17%) █
Preferencial            2,912 ( 0.51%) 
Empresarial                 3 ( 0.00%) 
TOTAL                 568,251 (100.00%)


### 4.1.3 Análisis de variable respuesta por segmento

In [28]:
df_resumen = crear_resumen_todos_segmentos(base_trtes)
print("\nRESUMEN DE TODOS LOS SEGMENTOS:")
print(df_resumen.to_string(index=False))


RESUMEN DE TODOS LOS SEGMENTOS:
     Segmento  Total  Ceros  % Ceros   Unos  % Unos
     Personal 361771 179698    49.67 182073   50.33
Personal plus 110014  54161    49.23  55853   50.77
    Micropyme  56746  33866    59.68  22880   40.32
       Social  18007  11313    62.83   6694   37.17
        Pymes  18798  14515    77.22   4283   22.78
 Preferencial   2912   1928    66.21    984   33.79
  Empresarial      3      2    66.67      1   33.33


In [29]:
print("\n\nANALIZANDO TODOS LOS SEGMENTOS:\n")
segmentos = base_trtes['segmento'].unique()

for segmento in segmentos:
    resultado = analizar_proporciones_segmento(
        base_trtes,
        segmento_valor=segmento,
        mostrar_detalle=True
    )



ANALIZANDO TODOS LOS SEGMENTOS:


ANÁLISIS DE PROPORCIONES - SEGMENTO: Personal

📊 PROPORCIONES GENERALES (TODA LA BASE):
--------------------------------------------------------------------------------
  Total de registros: 361,771
  Ceros (0): 179,698 casos → 49.67%
  Unos  (1): 182,073 casos → 50.33%

📅 PROPORCIONES POR FECHA:
--------------------------------------------------------------------------------
 Fecha  Total  Ceros  Unos  % Ceros  % Unos
202308  71366  35517 35849    49.77   50.23
202309  77525  37215 40310    48.00   52.00
202310  74643  38487 36156    51.56   48.44
202311  74666  36169 38497    48.44   51.56
202312  63571  32310 31261    50.83   49.17

--------------------------------------------------------------------------------
📈 ESTADÍSTICAS POR FECHA:
  Total de fechas: 5
  Promedio % ceros: 49.72%
  Promedio % unos:  50.28%
  Máximo % ceros:   51.56%
  Máximo % unos:    52.00%
  Mínimo % ceros:   48.00%
  Mínimo % unos:    48.44%


ANÁLISIS DE PROPORCIONES - S

### 4.1.4 Distribución de casos por producto

In [30]:
conteos_producto = base_trtes['producto'].value_counts()
total_producto = conteos_producto.sum()

print("\n" + "="*70)
print("DISTRIBUCIÓN DE CASOS POR PRODUCTO")
print("="*70)

for producto, cantidad in conteos_producto.items():
    porcentaje = (cantidad / total_producto * 100)
    barra = "█" * int(porcentaje / 2)
    print(f"{str(producto):20} {cantidad:8,} ({porcentaje:5.2f}%) {barra}")

print("="*70)
print(f"{'TOTAL':20} {total_producto:8,} (100.00%)")
print("="*70)


DISTRIBUCIÓN DE CASOS POR PRODUCTO
TARJETA DE CREDITO    243,422 (42.84%) █████████████████████
LIBRE INVERSION       201,430 (35.45%) █████████████████
ROTATIVOS              80,733 (14.21%) ███████
CARTERA ORDINARIA      11,916 ( 2.10%) █
LIBRANZA               10,340 ( 1.82%) 
HIPOTECARIO VIVIENDA   10,209 ( 1.80%) 
CARTERA MICROCREDITO    5,979 ( 1.05%) 
CREDIPAGO               1,407 ( 0.25%) 
SOBREGIRO               1,307 ( 0.23%) 
LEASING HABITACIONAL      649 ( 0.11%) 
TARJETAS DE CREDITO       228 ( 0.04%) 
CREDITOS DE CONSUMO       173 ( 0.03%) 
TESORERIA                 143 ( 0.03%) 
CREDIAGIL                 135 ( 0.02%) 
CREDITO A LA MANO          35 ( 0.01%) 
Titularizada               34 ( 0.01%) 
REESTRUCTURADO             27 ( 0.00%) 
LEASING                    23 ( 0.00%) 
VENTA DIGITAL              21 ( 0.00%) 
CUENTA CORRIENTE           17 ( 0.00%) 
OTROS HIPOTECARIO          13 ( 0.00%) 
LIBRANZA EX EMPLEADOS        6 ( 0.00%) 
MICROCREDITO                3 ( 0.00%

### 4.1.5 Análisis de variable respuesta por producto

In [31]:
df_resumen_producto = crear_resumen_todos_productos(base_trtes)
print("\nRESUMEN DE TODOS LOS PRODUCTOS:")
print(df_resumen_producto.to_string(index=False))


RESUMEN DE TODOS LOS PRODUCTOS:
             Producto  Total  Ceros  % Ceros   Unos  % Unos
   TARJETA DE CREDITO 243422 115245    47.34 128177   52.66
      LIBRE INVERSION 201430 107287    53.26  94143   46.74
            ROTATIVOS  80733  45502    56.36  35231   43.64
    CARTERA ORDINARIA  11916   7485    62.81   4431   37.19
             LIBRANZA  10340   7136    69.01   3204   30.99
 HIPOTECARIO VIVIENDA  10209   6986    68.43   3223   31.57
 LEASING HABITACIONAL    649    506    77.97    143   22.03
            CREDIPAGO   1407    826    58.71    581   41.29
 CARTERA MICROCREDITO   5979   3216    53.79   2763   46.21
            SOBREGIRO   1307   1087    83.17    220   16.83
            CREDIAGIL    135      0     0.00    135  100.00
              LEASING     23     18    78.26      5   21.74
            TESORERIA    143     77    53.85     66   46.15
  TARJETAS DE CREDITO    228     31    13.60    197   86.40
  CREDITOS DE CONSUMO    173     11     6.36    162   93.64
    CRE

In [32]:
print("\n\nANALIZANDO TODOS LOS PRODUCTOS:\n")
productos = base_trtes['producto'].unique()

for producto in productos:
    resultado = analizar_proporciones_producto(
        base_trtes,
        producto_valor=producto,
        mostrar_detalle=True
    )



ANALIZANDO TODOS LOS PRODUCTOS:


ANÁLISIS DE PROPORCIONES - PRODUCTO: TARJETA DE CREDITO

📊 PROPORCIONES GENERALES (TODA LA BASE):
--------------------------------------------------------------------------------
  Total de registros: 243,422
  Ceros (0): 115,245 casos → 47.34%
  Unos  (1): 128,177 casos → 52.66%

📅 PROPORCIONES POR FECHA:
--------------------------------------------------------------------------------
 Fecha  Total  Ceros  Unos  % Ceros  % Unos
202308  47877  21844 26033    45.63   54.37
202309  51877  23587 28290    45.47   54.53
202310  50917  25127 25790    49.35   50.65
202311  49371  22305 27066    45.18   54.82
202312  43380  22382 20998    51.60   48.40

--------------------------------------------------------------------------------
📈 ESTADÍSTICAS POR FECHA:
  Total de fechas: 5
  Promedio % ceros: 47.45%
  Promedio % unos:  52.55%
  Máximo % ceros:   51.60%
  Máximo % unos:    54.82%
  Mínimo % ceros:   45.18%
  Mínimo % unos:    48.40%


ANÁLISIS DE PROPOR

### 4.1.6 Análisis de clientes recurrentes

In [33]:
resultado = mapear_clientes_recurrentes(
    base_trtes,
    col_nit='nit_enmascarado',
    col_fecha='fecha_var_rpta_alt',
    min_fechas=2,
    mostrar_detalle=True
)

if resultado:
    print("\n>>> Clientes más recurrentes:")
    print(resultado['clientes_recurrentes'].head(10))
    
    print("\n>>> Cruce NIT x Fecha (muestra):")
    print(resultado['detalle_cruce'].head(20))


MAPEO DE CLIENTES RECURRENTES - SEGMENTO: TODOS

📊 ESTADÍSTICAS GENERALES:
----------------------------------------------------------------------------------------------------
  Total clientes únicos: 267,256
  Clientes recurrentes: 96,686 (36.18%)
  Total registros: 568,251
  Fechas diferentes: 5
  Promedio fechas por cliente: 1.51
  Máximas fechas de un cliente: 5
  Promedio registros por cliente: 2.13

📈 DISTRIBUCIÓN DE FRECUENCIA:
----------------------------------------------------------------------------------------------------
  Clientes con 1 fecha(s): 170570 (63.82%) ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

### 4.1.7 Conclusiones del análisis exploratorio inicial

#### 🔍 Aspectos relevantes identificados

**Variable Respuesta:**
- La distribución de ceros y unos en la variable respuesta no presenta un desbalanceo considerable, lo que permite construir un modelo sin necesidad de recurrir a técnicas de balanceo inicialmente.

**Segmentación de Clientes:**
- La mayor proporción de datos corresponde al segmento **Personal**, lo que puede generar cierto sesgo en el modelo. Este sesgo puede mitigarse si los patrones de comportamiento son similares entre segmentos.
- Los diferentes segmentos mantienen proporciones balanceadas de la variable respuesta, salvo en casos particulares con menor cantidad de registros.

**Distribución de Productos:**
- Existe una concentración marcada en tres productos: **Tarjeta de Crédito**, **Libre Inversión** y **Rotativos**, que representan el 92.5% de la base.
- Estos tres productos principales mantienen proporciones balanceadas entre ceros y unos.
- Los demás productos tienen tamaños mucho menores y algunas proporciones extremas (0% o 100%).

**Característica de Clientes Recurrentes:**
- Se identifican clientes que aparecen en múltiples fechas de análisis, aspecto relevante para la metodología de separación de datos de entrenamiento y validación.
- Esta característica es crítica para garantizar una medición adecuada del desempeño del modelo.

---

#### 🎯 Conclusiones sobre el desempeño esperado del modelo

Basándonos en los aspectos anteriormente descritos, podemos plantear preliminarmente que el modelo entrenado puede tener un desempeño considerablemente mejor en:

- **Población de clientes del segmento Personal** (la más representada en la base)
- **Productos principales**: Tarjeta de Crédito, Libre Inversión y Rotativos

Sin embargo, esto no significa que el desempeño sea malo para los demás segmentos o productos. Los patrones de comportamiento pueden ser similares entre categorías, permitiendo generalización del modelo a otras grupos.

---

#### ⚠️ Consideraciones para la estrategia de separación de datos

Teniendo en cuenta las distribuciones identificadas (por segmento, producto, variable respuesta y clientes), es **crítico establecer una estrategia cuidadosa de split** para garantizar que el modelo:

1. **Mantenga el balance de la variable respuesta** en entrenamiento y validación para entrenar correctamente
2. **Preserve las proporciones** de segmentos y productos definidas en el análisis
3. **Separe completamente los clientes** entre base de entrenamiento y prueba (sin solapamiento) para garantizar una medición objetiva

---

#### 💡 Enfoque de desarrollo propuesto

El modelo desarrollado debe ser **muy bueno en la segmentación general** (discriminar entre ceros y unos), pero no necesariamente excelente en predicción específica por segmento o producto. Esta es una opción de diseño que busca robustez general sobre especialización local.

### 4.2 Exploratorio variables predictoras

### 4.2.1 Exploratorio variables modelos cobranza

In [34]:
base_info_modelos = pd.read_csv(ins / insumos['base_info_modelos']['archivo'])

In [35]:
base_info_modelos.head(10)

,nit_enmascarado,num_oblig_enmascarado,fecha_corte,lote,prob_propension,prob_alrt_temprana,prob_auto_cura
0,296482,102381,202308,1,0.761350,0.193744,0.684784
1,391957,742315,202310,2,0.741803,0.384184,0.483696
2,229894,359919,202307,1,0.835373,0.285157,0.826225
3,478963,239064,202303,3,0.445002,0.629652,0.346508
4,349609,923348,202308,2,0.784365,0.419123,0.667603
5,102595,857453,202307,1,0.742234,0.364649,0.527945
6,508610,638552,202312,2,0.382216,0.606773,0.176670
7,307136,561298,202306,1,0.806566,0.765648,0.533541
8,332328,1031223,202305,3,0.435027,0.569402,0.268361
9,411179,969322,202309,1,0.857433,0.184038,0.877824


In [36]:
base_info_modelos.describe()

,nit_enmascarado,num_oblig_enmascarado,fecha_corte,lote,prob_propension,prob_alrt_temprana,prob_auto_cura
count,4.804836e+06,4.804836e+06,4.804836e+06,4.804836e+06,4.804836e+06,4.804829e+06,4.804829e+06
mean,3.085411e+05,5.597008e+05,2.023067e+05,1.568148e+00,7.002381e-01,4.349893e-01,4.877742e-01
std,1.828653e+05,3.104282e+05,3.369213e+00,7.002282e-01,2.140671e-01,2.277477e-01,2.501165e-01
min,1.000000e+00,3.000000e+00,2.023010e+05,1.000000e+00,3.432506e-02,2.300400e-02,4.604448e-02
25%,1.526200e+05,2.955510e+05,2.023040e+05,1.000000e+00,6.286394e-01,2.222855e-01,2.581266e-01
50%,2.999030e+05,5.749985e+05,2.023070e+05,1.000000e+00,7.739288e-01,4.877411e-01,4.441285e-01
75%,4.652200e+05,8.269940e+05,2.023100e+05,2.000000e+00,8.392768e-01,6.158977e-01,7.501535e-01
max,6.347090e+05,1.080719e+06,2.023120e+05,3.000000e+00,9.593788e-01,9.318064e-01,9.512819e-01


In [37]:
dup_count = base_info_modelos.groupby(['nit_enmascarado','num_oblig_enmascarado', 'fecha_corte']).size().reset_index(name='count')
duplicados = dup_count[dup_count['count'] > 1].sort_values('count', ascending=False)
duplicados

,nit_enmascarado,num_oblig_enmascarado,fecha_corte,count


### 4.2.2 Exploratorio variables sociodemográficas

In [38]:
base_info_socio = pd.read_csv(ins / insumos['base_info_socio']['archivo'])

In [39]:
base_info_socio.head(10)

,nit_enmascarado,cod_tipo_doc,tipo_cli,ctrl_terc,genero_cli,ano_nac_cli,edad_cli,estado_civil,tipo_vivienda,num_hijos,...,nicho,region_of,nombre_dpto_dirp,egresos_mes,tot_patrimonio,ciiu,smmlv,year,month,ingestion_day
0,536377,1,PERSONA NATURAL,CLIENTE,F,1998.0,25.0,SOLTERO,FAMILIAR,0.0,...,MUJERES,CENTRO,SANTANDER,0.0,33750000.0,NaN,1160000.0,2023,12,31
1,512257,1,PERSONA NATURAL,CLIENTE,F,1984.0,38.0,UNION LIBRE,PROPIA,0.0,...,MUJERES,CARIBE,MAGDALENA,250000.0,5000000.0,ASALARIADOS,1160000.0,2023,9,30
2,5788,1,PERSONA NATURAL,CLIENTE SOCIAL,M,1960.0,63.0,NaN,NaN,0.0,...,NaN,DIRECCIÓN GENERAL,SIN INFORMACION,0.0,0.0,ASALARIADOS,1160000.0,2023,11,30
3,132245,1,PERSONA NATURAL,CLIENTE,F,1982.0,41.0,CASADO,NaN,1.0,...,MUJERES,SUR,VALLE,1000000.0,62513000.0,ASALARIADOS,1160000.0,2023,7,31
4,245279,1,PERSONA NATURAL,CLIENTE,M,1980.0,43.0,CASADO,ALQUILADA,0.0,...,NaN,BOGOTA Y CUNDINAMARCA,CALDAS,1000000.0,34135000.0,ASALARIADOS,1160000.0,2023,12,31
5,190778,1,PERSONA NATURAL,CLIENTE,F,1979.0,43.0,UNION LIBRE,FAMILIAR,1.0,...,NaN,BOGOTA Y CUNDINAMARCA,META,300000.0,1.0,"COMERCIO AL POR MENOR DE ALIMENTOS, BEBIDAS Y ...",1160000.0,2023,7,31
6,456849,1,PERSONA NATURAL,CLIENTE,F,1989.0,34.0,SOLTERO,NaN,0.0,...,MUJERES,CARIBE,BOLÍVAR,100000.0,6896000.0,ASALARIADOS,1160000.0,2023,8,31
7,552797,1,PERSONA NATURAL,CLIENTE,M,1994.0,29.0,SOLTERO,NaN,0.0,...,NaN,BOGOTA Y CUNDINAMARCA,TOLIMA,200000.0,5000000.0,RENTISTAS DE CAPITAL SÓLO PARA PERSONAS NATURALES,1160000.0,2023,9,30
8,389195,1,PERSONA NATURAL,CLIENTE,M,1993.0,30.0,SOLTERO,NaN,0.0,...,NaN,ANTIOQUIA,ANTIOQUIA,450000.0,30365744.0,ASALARIADOS,1160000.0,2023,12,31
9,441372,1,PERSONA NATURAL,CLIENTE,F,1999.0,23.0,SOLTERO,NaN,0.0,...,MUJERES,ANTIOQUIA,ANTIOQUIA,92000.0,5507000.0,ASALARIADOS,1160000.0,2023,8,31


In [40]:
base_info_socio.columns

Index(['nit_enmascarado', 'cod_tipo_doc', 'tipo_cli', 'ctrl_terc',
       'genero_cli', 'ano_nac_cli', 'edad_cli', 'estado_civil',
       'tipo_vivienda', 'num_hijos', 'personas_dependientes',
       'nivel_academico', 'ocup', 'act_econom', 'sector', 'subsector',
       'declarante', 'total_ing', 'tot_activos', 'tot_pasivos',
       'origen_fondos', 'f_vinc', 'f_ult_mantenimiento', 'canal_actualizacion',
       'cli_actualizado', 'segm', 'subsegm', 'nicho', 'region_of',
       'nombre_dpto_dirp', 'egresos_mes', 'tot_patrimonio', 'ciiu', 'smmlv',
       'year', 'month', 'ingestion_day'],
      dtype='object')

In [41]:
dup_count = base_info_socio.groupby(['nit_enmascarado','year', 'month', 'ingestion_day']).size().reset_index(name='count')
duplicados = dup_count[dup_count['count'] > 1].sort_values('count', ascending=False)
duplicados

,nit_enmascarado,year,month,ingestion_day,count
240958,348225,2023,8,31,2


In [42]:
base_info_socio[(base_info_socio["nit_enmascarado"]==348225) & ((base_info_socio["year"]*10000+base_info_socio["month"]*100+base_info_socio["ingestion_day"])==20230831)]

,nit_enmascarado,cod_tipo_doc,tipo_cli,ctrl_terc,genero_cli,ano_nac_cli,edad_cli,estado_civil,tipo_vivienda,num_hijos,...,nicho,region_of,nombre_dpto_dirp,egresos_mes,tot_patrimonio,ciiu,smmlv,year,month,ingestion_day
202921,348225,4,PERSONA NATURAL,EXCLIENTE,M,2003.0,20.0,SOLTERO,NaN,0.0,...,NaN,BANCO,BOGOTÁ,500000.0,0.0,NaN,1160000.0,2023,8,31
385445,348225,1,PERSONA NATURAL,CLIENTE,M,1990.0,33.0,NaN,NaN,0.0,...,NaN,BOGOTA Y CUNDINAMARCA,NaN,0.0,0.0,NaN,1160000.0,2023,8,31


In [43]:
base_info_socio[(base_info_socio["nit_enmascarado"]==348225) & ((base_info_socio["year"]*10000+base_info_socio["month"]*100+base_info_socio["ingestion_day"])==20230831)][["nit_enmascarado", "segm"]]

,nit_enmascarado,segm
202921,348225,PERSONAL
385445,348225,PERSONAL


**Nota**: Dado que exite dos registros duplicados para el numero de indentificación 348225 en la fecha de referencia 20230831, se realiza la eliminación de uno de los registros para evitar duplicados al momento de cruzar la información. En el caso particular de estudio se procede a eliminar el registro con el cod_tipo_doc = 4.

In [44]:
base_info_socio = base_info_socio.drop(202921)

**Nota**: Finalmente, dado que estamos realizando un proceso de analisis a nivel mensual por cliente y obligacion, se toma como referencia la información mas reciente para el cliente. En un escenario real se tomaria la información del ultimo dia del mes, o en dado caso buscando el registro mas cercano a la fecha de analisis, no obstante para efectos prácticos se toma el ultimo registro disponible para limitar la cantidad de nulos y simplificar el proceso.

In [45]:
base_info_socio['fecha_temp'] = base_info_socio['year']*10000 + base_info_socio['month']*100 + base_info_socio['ingestion_day']
base_info_socio = base_info_socio.loc[base_info_socio.groupby('nit_enmascarado')['fecha_temp'].idxmax()]
base_info_socio = base_info_socio.drop('fecha_temp', axis=1)

In [46]:
base_info_socio["fecha_corte"] = base_info_socio["year"]*100+base_info_socio["month"]
base_info_socio = base_info_socio.drop(['year', 'month', 'ingestion_day'], axis=1)

In [47]:
base_info_socio[(base_info_socio["nit_enmascarado"]==630611)]

,nit_enmascarado,cod_tipo_doc,tipo_cli,ctrl_terc,genero_cli,ano_nac_cli,edad_cli,estado_civil,tipo_vivienda,num_hijos,...,segm,subsegm,nicho,region_of,nombre_dpto_dirp,egresos_mes,tot_patrimonio,ciiu,smmlv,fecha_corte
105369,630611,1,PERSONA NATURAL,CLIENTE,F,1998.0,25.0,SOLTERO,NaN,0.0,...,PERSONAL,MEDIO,MUJERES,ANTIOQUIA,ANTIOQUIA,100000.0,165012.0,NaN,1160000.0,202311


### 4.2.3 Exploratorio variables pagos

In [48]:
base_info_pagos = pd.read_csv(ins / insumos['base_info_pagos']['archivo'])

In [49]:
base_info_pagos.head(10)

,nit_enmascarado,num_oblig_enmascarado,fecha_corte,producto,aplicativo,segmento,valor_cuota_mes,pago_total,fecha_pago_minima,fecha_pago_maxima,porc_pago,marca_pago,ajustes_banco
0,482906,362297,20230731,CARTERA MICROCREDITO,L,MICROPYME,311950.0,1862788.0,20230621.0,20230721.0,597.0,PAGO_MENOS,NO
1,121735,186855,20230228,LIBRE INVERSION,L,SOCIAL,131030.0,264157.0,20230127.0,20230207.0,202.0,PAGO_MENOS,NO
2,582719,675503,20231031,TARJETA DE CREDITO,K,PERSONAL,1405339.0,3320.0,20231003.0,20231003.0,0.0,PAGO_MENOS,NO
3,299903,107931,20230228,ROTATIVOS,L,PERSONAL PLUS,19220.0,0.0,NaN,NaN,0.0,FACTURACION_MES_SGTE,NO
4,88625,566060,20230430,ROTATIVOS,L,PERSONAL,384684.0,769368.0,20230403.0,20230403.0,200.0,PAGO_MAS,NO
5,25029,324131,20230630,LIBRE INVERSION,L,PREFERENCIAL,2232058.0,100000.0,20230531.0,20230531.0,4.0,IGUAL,NO
6,271030,105944,20230430,LIBRE INVERSION,L,PERSONAL PLUS,1403999.0,5602093.0,20230330.0,20230331.0,399.0,PAGO_MAS,NO
7,535835,639317,20231231,TARJETA DE CREDITO,K,PERSONAL,395828.0,0.0,NaN,NaN,0.0,NO_PAGO,NO
8,497594,685488,20230531,TARJETA DE CREDITO,K,PERSONAL,1031953.0,2063908.0,20230511.0,20230511.0,200.0,PAGO_MAS,NO
9,296985,846060,20230131,TARJETA DE CREDITO,V,PERSONAL PLUS,1237998.0,1240000.0,20221227.0,20221227.0,100.0,IGUAL,NO


In [50]:
base_info_pagos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4855035 entries, 0 to 4855034
Data columns (total 13 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   nit_enmascarado        int64  
 1   num_oblig_enmascarado  int64  
 2   fecha_corte            int64  
 3   producto               object 
 4   aplicativo             object 
 5   segmento               object 
 6   valor_cuota_mes        float64
 7   pago_total             float64
 8   fecha_pago_minima      float64
 9   fecha_pago_maxima      float64
 10  porc_pago              float64
 11  marca_pago             object 
 12  ajustes_banco          object 
dtypes: float64(5), int64(3), object(5)
memory usage: 481.5+ MB


In [51]:
dup_count = base_info_pagos.groupby(['nit_enmascarado',
                                'num_oblig_enmascarado', 
                                'fecha_corte']).size().reset_index(name='count')
duplicados = dup_count[dup_count['count'] > 1].sort_values('count', ascending=False)
duplicados

,nit_enmascarado,num_oblig_enmascarado,fecha_corte,count
70273,8154,620951,20231130,2
497698,60968,883013,20231130,2
1492665,186030,649788,20231130,2
1561865,194625,475936,20231130,2
1636940,204090,879508,20231130,2
1814370,226143,768774,20231130,2
1957277,243368,1072018,20231130,2
2592439,318753,1052330,20231130,2
2952045,372957,612587,20230630,2
3577775,456315,913564,20231130,2


In [52]:
base_info_pagos[(base_info_pagos["nit_enmascarado"]==8154) & (base_info_pagos["num_oblig_enmascarado"]==620951) & (base_info_pagos["fecha_corte"]==20231130)]

,nit_enmascarado,num_oblig_enmascarado,fecha_corte,producto,aplicativo,segmento,valor_cuota_mes,pago_total,fecha_pago_minima,fecha_pago_maxima,porc_pago,marca_pago,ajustes_banco
765400,8154,620951,20231130,TARJETA DE CREDITO,K,PERSONAL,293868.0,185584.0,20231025.0,20231030.0,63.0,PAGO_MENOS,NO
2346362,8154,620951,20231130,TARJETA DE CREDITO,K,PERSONAL,293868.0,185585.0,20231025.0,20231030.0,63.0,PAGO_MENOS,NO


In [53]:
base_info_pagos[(base_info_pagos["nit_enmascarado"]==60968) & (base_info_pagos["num_oblig_enmascarado"]==883013) & (base_info_pagos["fecha_corte"]==20231130)]

,nit_enmascarado,num_oblig_enmascarado,fecha_corte,producto,aplicativo,segmento,valor_cuota_mes,pago_total,fecha_pago_minima,fecha_pago_maxima,porc_pago,marca_pago,ajustes_banco
1274285,60968,883013,20231130,TARJETA DE CREDITO,V,PERSONAL PLUS,4024572.0,5736608.0,20231017.0,20231019.0,143.0,PAGO_MENOS,NO
2814156,60968,883013,20231130,TARJETA DE CREDITO,V,PERSONAL PLUS,4024572.0,5736607.0,20231017.0,20231019.0,143.0,PAGO_MENOS,NO


**Nota:** Como puede apreciarse la base tiene presencia de datos duplicados, lo cual afecta directamente el proceso de unificación de datos, teniendo en cuenta que las diferencias en los datos se encuentran principalmente con diferencias con el valor del pago total, se procede a unificar la información y a calcular un valor promedio entre los valores de pago total

In [54]:
base_info_pagos_agg = base_info_pagos.groupby([
    'nit_enmascarado', 
    'num_oblig_enmascarado', 
    'fecha_corte', 
    'producto', 
    'aplicativo', 
    'segmento', 
    'valor_cuota_mes', 
    'fecha_pago_minima', 
    'fecha_pago_maxima', 
    'porc_pago', 
    'marca_pago', 
    'ajustes_banco'
], as_index=False).agg(pago_total=('pago_total', 'mean'))

In [55]:
base_info_pagos=base_info_pagos_agg[['nit_enmascarado', 
    'num_oblig_enmascarado', 
    'fecha_corte', 
    'producto', 
    'aplicativo', 
    'segmento', 
    'valor_cuota_mes', 
    'pago_total',
    'fecha_pago_minima', 
    'fecha_pago_maxima', 
    'porc_pago', 
    'marca_pago', 
    'ajustes_banco'
    ]]

In [56]:
base_info_pagos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3457240 entries, 0 to 3457239
Data columns (total 13 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   nit_enmascarado        int64  
 1   num_oblig_enmascarado  int64  
 2   fecha_corte            int64  
 3   producto               object 
 4   aplicativo             object 
 5   segmento               object 
 6   valor_cuota_mes        float64
 7   pago_total             float64
 8   fecha_pago_minima      float64
 9   fecha_pago_maxima      float64
 10  porc_pago              float64
 11  marca_pago             object 
 12  ajustes_banco          object 
dtypes: float64(5), int64(3), object(5)
memory usage: 342.9+ MB


In [57]:
dup_count = base_info_pagos.groupby(['nit_enmascarado',
                                'num_oblig_enmascarado', 
                                'fecha_corte']).size().reset_index(name='count')
duplicados = dup_count[dup_count['count'] > 1].sort_values('count', ascending=False)
duplicados

,nit_enmascarado,num_oblig_enmascarado,fecha_corte,count
2090092,372957,612587,20230630,2
2562026,459810,1063283,20230731,2


In [58]:
base_info_pagos[(base_info_pagos["nit_enmascarado"]==372957) & (base_info_pagos["num_oblig_enmascarado"]==612587) & (base_info_pagos["fecha_corte"]==20230630)]

,nit_enmascarado,num_oblig_enmascarado,fecha_corte,producto,aplicativo,segmento,valor_cuota_mes,pago_total,fecha_pago_minima,fecha_pago_maxima,porc_pago,marca_pago,ajustes_banco
2090092,372957,612587,20230630,HIPOTECARIO VIVIENDA,4,PERSONAL PLUS,1796304.0,0.0,20230615.0,20230615.0,0.0,IGUAL,NO
2090093,372957,612587,20230630,HIPOTECARIO VIVIENDA,4,PERSONAL PLUS,1796388.0,0.0,20230615.0,20230615.0,0.0,IGUAL,NO


In [59]:
base_info_pagos[(base_info_pagos["nit_enmascarado"]==459810) & (base_info_pagos["num_oblig_enmascarado"]==1063283) & (base_info_pagos["fecha_corte"]==20230731)]

,nit_enmascarado,num_oblig_enmascarado,fecha_corte,producto,aplicativo,segmento,valor_cuota_mes,pago_total,fecha_pago_minima,fecha_pago_maxima,porc_pago,marca_pago,ajustes_banco
2562027,459810,1063283,20230731,TARJETA DE CREDITO,M,PERSONAL PLUS,1317225.0,2701740.0,20230622.0,20230622.0,205.0,PAGO_MAS,NO
2562028,459810,1063283,20230731,TARJETA DE CREDITO,M,PERSONAL PLUS,2630793.0,2701740.0,20230622.0,20230622.0,103.0,PAGO_MAS,NO


**Nota:** Notese que despues de aplicar el agrupamiento de los datos `base_info_pagos`, se mantiene la duplicidad en dos registros específicos, esto a raiz de variaciones en los valores de valor cuota mes y porcentaje de pago, esto puede tratarse de errores derivados en la base o pequeñas variaciones en los datos como en el caso anterior, para no extender el proceso de limpieza se decide eliminar de forma aleatoria uno de los registros duplicados.

In [60]:
id_e1 = [2090092, 2090093]
id_e1 = random.choice(id_e1)
id_e2 = [2562027, 2562028]
id_e2 = random.choice(id_e2)
base_info_pagos = base_info_pagos.drop(id_e1)
base_info_pagos = base_info_pagos.drop(id_e2)
base_info_pagos.reset_index(drop=True, inplace=True)

In [61]:
base_info_pagos['ano_mes'] = base_info_pagos['fecha_corte'] // 100  
base_info_pagos = base_info_pagos.loc[
    base_info_pagos.groupby(['nit_enmascarado', 'num_oblig_enmascarado', 'ano_mes'])['fecha_corte'].idxmax()
].reset_index(drop=True)
base_info_pagos = base_info_pagos.drop('ano_mes', axis=1)
base_info_pagos["fecha_corte"] = base_info_pagos['fecha_corte'] // 100

### 4.2.4 Unificación de fuentes

In [62]:
resultado = base_trtes.merge(
    base_info_modelos, 
    left_on=['nit_enmascarado', 'num_oblig_enmascarado', 'fecha_analisis'],
    right_on=['nit_enmascarado', 'num_oblig_enmascarado', 'fecha_corte'],
    how='left',
    suffixes=('', '_modelos') 
)
resultado = resultado.merge(
    base_info_socio, 
    left_on=['nit_enmascarado'],
    right_on=['nit_enmascarado'],
    how='left',
    suffixes=('', '_socio')
)
resultado = resultado.merge(
    base_info_pagos, 
    left_on=['nit_enmascarado', 'num_oblig_enmascarado', 'fecha_analisis'],
    right_on=['nit_enmascarado', 'num_oblig_enmascarado', 'fecha_corte'],
    how='left',
    suffixes=('', '_pagos')
)
# resultado = resultado.loc[:, ~resultado.columns.str.endswith(('_modelos', '_socio', '_pagos'))]

In [63]:
resultado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568251 entries, 0 to 568250
Data columns (total 100 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   nit_enmascarado                  568251 non-null  int64  
 1   num_oblig_orig_enmascarado       568251 non-null  int64  
 2   num_oblig_enmascarado            568251 non-null  int64  
 3   fecha_var_rpta_alt               568251 non-null  int64  
 4   var_rpta_alt                     568251 non-null  int64  
 5   tipo_var_rpta_alt                568251 non-null  object 
 6   banca                            568251 non-null  object 
 7   segmento                         568251 non-null  object 
 8   producto                         568251 non-null  object 
 9   producto_cons                    568251 non-null  object 
 10  aplicativo                       568251 non-null  object 
 11  min_mora                         568251 non-null  int64  
 12  m

In [64]:
resultado.head(100).to_excel('primeros_100_registros.xlsx', index=False)

### 4.2.5 Filtrado de información

In [65]:
base_oot = pd.read_csv(ins / insumos['base_oot']['archivo'])

In [66]:
base_oot.head(10)

,nit_enmascarado,num_oblig_orig_enmascarado,num_oblig_enmascarado,fecha_var_rpta_alt
0,257335,444821,635511,202401
1,59584,350400,730364,202401
2,397604,973821,106521,202401
3,368086,382995,696856,202401
4,255009,434238,645924,202401
5,417751,109691,978105,202401
6,15170,695103,385239,202401
7,64083,1033680,46662,202401
8,522913,59984,1044995,202401
9,46441,633313,447029,202401


In [67]:
base_oot['fecha_referencia_1'] = pd.to_datetime(base_oot['fecha_var_rpta_alt'].astype(str) + '01', format='%Y%m%d')
base_oot['fecha_referencia_2'] = base_oot['fecha_referencia_1'].apply(lambda x: x - relativedelta(months=1))
base_oot['fecha_analisis'] = base_oot['fecha_referencia_2'].dt.strftime('%Y%m').astype(int)
base_oot = base_oot.drop(['fecha_referencia_1', 'fecha_referencia_2'], axis=1)

**Nota:** Notese en este caso que los datos de `base_oot`, no contienen la información de la `base_trtes`, esto implica que para efectos del ejercicio debemos descartar todas las variables de la base `base_trtes`, excluyendo la variable respuesta, esto debido a que si alguna de estas variables es seleccionada por el modelo no se podra realizar el punteo sobre `base_oot`. Las variables que se descartan inicalmente contienen información de segemento, producto, mora gestiones entre otras las cuales no se pueden tener en consideración en este caso, sin embargo algunas de estas variables se encuentran dentro de las otras bases disponibles.

In [68]:
columnas_trtes=[
'banca',
'segmento',
'producto',
'producto_cons',
'aplicativo',
'min_mora',
'max_mora',
'dias_mora_fin',
'rango_mora',
'vlr_obligacion',
'vlr_vencido',
'saldo_capital',
'endeudamiento',
'desc_alternativa1',
'desc_alternativa2',
'desc_alternativa3',
'cant_alter_posibles',
'alter_posible1_2',
'alter_posible2_2',
'alter_posible3_2',
'cant_gestiones',
'cant_gestiones_binario',
'rpc',
'promesas_cumplidas',
'cant_promesas_cumplidas_binario',
'cant_acuerdo',
'cant_acuerdo_binario',
'descripcion_ranking_mejor_ult',
'descripcion_ranking_post_ult',
'marca_alt_rank',
'marca_alt_apli',
'valor_cuota_mes',
'pago_cuota',
'porc_pago_cuota',
'pago_mes',
'porc_pago_mes',
'pagos_tanque',
'marca_debito_mora',
'alternativa_aplicada_agr',
'marca_agrupada_rgo',
'marca_pago',
'marca_alternativa',
'marca_alternativa_orig',
]
resultado.drop(columns=columnas_trtes, inplace=True)

**Nota:** Adicionalmente es necesario borrar las columnas con información redundate de las bases y las que fueron generados durante el cruce.

In [69]:
columnas_trtes=[
'fecha_corte',
'lote',
'fecha_corte_socio',
'fecha_corte_pagos',
'segmento_pagos'
]
resultado.drop(columns=columnas_trtes, inplace=True)

In [70]:
resultado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568251 entries, 0 to 568250
Data columns (total 52 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   nit_enmascarado             568251 non-null  int64  
 1   num_oblig_orig_enmascarado  568251 non-null  int64  
 2   num_oblig_enmascarado       568251 non-null  int64  
 3   fecha_var_rpta_alt          568251 non-null  int64  
 4   var_rpta_alt                568251 non-null  int64  
 5   tipo_var_rpta_alt           568251 non-null  object 
 6   fecha_analisis              568251 non-null  int32  
 7   prob_propension             566363 non-null  float64
 8   prob_alrt_temprana          566363 non-null  float64
 9   prob_auto_cura              566363 non-null  float64
 10  cod_tipo_doc                456290 non-null  float64
 11  tipo_cli                    456290 non-null  object 
 12  ctrl_terc                   456290 non-null  object 
 13  genero_cli    

In [71]:
col_id = ['nit_enmascarado','num_oblig_orig_enmascarado','num_oblig_enmascarado','fecha_analisis']
col_pred = ['fecha_var_rpta_alt','var_rpta_alt','tipo_var_rpta_alt']
col_vars = [col for col in resultado.columns if col not in col_id + col_pred]
resultado = resultado[col_id + col_pred + col_vars]
resultado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568251 entries, 0 to 568250
Data columns (total 52 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   nit_enmascarado             568251 non-null  int64  
 1   num_oblig_orig_enmascarado  568251 non-null  int64  
 2   num_oblig_enmascarado       568251 non-null  int64  
 3   fecha_analisis              568251 non-null  int32  
 4   fecha_var_rpta_alt          568251 non-null  int64  
 5   var_rpta_alt                568251 non-null  int64  
 6   tipo_var_rpta_alt           568251 non-null  object 
 7   prob_propension             566363 non-null  float64
 8   prob_alrt_temprana          566363 non-null  float64
 9   prob_auto_cura              566363 non-null  float64
 10  cod_tipo_doc                456290 non-null  float64
 11  tipo_cli                    456290 non-null  object 
 12  ctrl_terc                   456290 non-null  object 
 13  genero_cli    

In [72]:
resultado_nulos = calcular_porcentaje_nulos(resultado, columnas=col_vars, umbral=60.0)
print("Resumen de nulos por columna:")
print(resultado_nulos['resumen'].to_string(index=False))
print(f"\nColumnas a descartar (>60% nulos): {len(resultado_nulos['columnas_descartar'])}")
print(resultado_nulos['columnas_descartar'])

Resumen de nulos por columna:
              columna  cantidad_nulos  porcentaje_nulos
        tipo_vivienda          419001             73.74
      nivel_academico          364075             64.07
                nicho          352148             61.97
        ajustes_banco          233470             41.09
       producto_pagos          233470             41.09
     aplicativo_pagos          233470             41.09
valor_cuota_mes_pagos          233470             41.09
           pago_total          233470             41.09
    fecha_pago_minima          233470             41.09
    fecha_pago_maxima          233470             41.09
            porc_pago          233470             41.09
     marca_pago_pagos          233470             41.09
  canal_actualizacion          156198             27.49
         estado_civil          146952             25.86
                 ciiu          141520             24.90
               sector          141520             24.90
            subsec

In [73]:
resultado = resultado.drop(columns=resultado_nulos['columnas_descartar'])
col_vars = [col for col in col_vars if col not in resultado_nulos['columnas_descartar']]

In [74]:
resultado_concurrencia = calcular_concurrencia_valores(
    resultado, columnas=col_vars, umbral=80.0,
    incluir_enteros=True, max_valores_unicos_enteros=50
)
print("Resumen (valor dominante por columna categórica):")
print(resultado_concurrencia['resumen'].to_string(index=False))

Resumen (valor dominante por columna categórica):
              columna                 valor_dominante  porcentaje_dominante  n_valores_unicos
                smmlv                       1160000.0                100.00                 1
        ajustes_banco                              NO                 98.42                 3
             tipo_cli                 PERSONA NATURAL                 97.42                 2
         cod_tipo_doc                             1.0                 96.97                 5
            ctrl_terc                         CLIENTE                 96.96                 6
           declarante                               N                 89.53                 2
            num_hijos                             0.0                 88.54                24
personas_dependientes                             0.0                 87.08                26
               sector                        PERSONAS                 85.03                10
          

In [75]:
print(resultado_concurrencia['detalle']['cli_actualizado'])

  valor  frecuencia  porcentaje
0     N      263202       57.68
1     S      193088       42.32


In [76]:
resultado = resultado.drop(columns=resultado_concurrencia['columnas_descartar'])
col_vars = [col for col in col_vars if col not in resultado_concurrencia['columnas_descartar']]

In [77]:
resultado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568251 entries, 0 to 568250
Data columns (total 37 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   nit_enmascarado             568251 non-null  int64  
 1   num_oblig_orig_enmascarado  568251 non-null  int64  
 2   num_oblig_enmascarado       568251 non-null  int64  
 3   fecha_analisis              568251 non-null  int32  
 4   fecha_var_rpta_alt          568251 non-null  int64  
 5   var_rpta_alt                568251 non-null  int64  
 6   tipo_var_rpta_alt           568251 non-null  object 
 7   prob_propension             566363 non-null  float64
 8   prob_alrt_temprana          566363 non-null  float64
 9   prob_auto_cura              566363 non-null  float64
 10  genero_cli                  444489 non-null  object 
 11  ano_nac_cli                 444503 non-null  float64
 12  edad_cli                    444503 non-null  float64
 13  estado_civil  

In [78]:
len(col_vars)

30

In [79]:
resultado.head(100).to_excel('primeros_100_registros.xlsx', index=False)

### 4.2.6 Generación de base final

In [80]:
resultado_oot = base_oot.merge(
    base_info_modelos, 
    left_on=['nit_enmascarado', 'num_oblig_enmascarado', 'fecha_analisis'],
    right_on=['nit_enmascarado', 'num_oblig_enmascarado', 'fecha_corte'],
    how='left',
    suffixes=('', '_modelos') 
)
resultado_oot = resultado_oot.merge(
    base_info_socio, 
    left_on=['nit_enmascarado'],
    right_on=['nit_enmascarado'],
    how='left',
    suffixes=('', '_socio')
)
resultado_oot = resultado_oot.merge(
    base_info_pagos, 
    left_on=['nit_enmascarado', 'num_oblig_enmascarado', 'fecha_analisis'],
    right_on=['nit_enmascarado', 'num_oblig_enmascarado', 'fecha_corte'],
    how='left',
    suffixes=('', '_pagos')
)

In [81]:
len(resultado_oot)

112549

In [82]:
len(base_oot)

112549

In [83]:
resultado_oot["var_rpta_alt"] = pd.NA
resultado_oot["var_rpta_alt"] = resultado_oot["var_rpta_alt"].astype("Int64")
resultado_oot["tipo_var_rpta_alt"]=""

In [84]:
resultado_oot.columns

Index(['nit_enmascarado', 'num_oblig_orig_enmascarado',
       'num_oblig_enmascarado', 'fecha_var_rpta_alt', 'fecha_analisis',
       'fecha_corte', 'lote', 'prob_propension', 'prob_alrt_temprana',
       'prob_auto_cura', 'cod_tipo_doc', 'tipo_cli', 'ctrl_terc', 'genero_cli',
       'ano_nac_cli', 'edad_cli', 'estado_civil', 'tipo_vivienda', 'num_hijos',
       'personas_dependientes', 'nivel_academico', 'ocup', 'act_econom',
       'sector', 'subsector', 'declarante', 'total_ing', 'tot_activos',
       'tot_pasivos', 'origen_fondos', 'f_vinc', 'f_ult_mantenimiento',
       'canal_actualizacion', 'cli_actualizado', 'segm', 'subsegm', 'nicho',
       'region_of', 'nombre_dpto_dirp', 'egresos_mes', 'tot_patrimonio',
       'ciiu', 'smmlv', 'fecha_corte_socio', 'fecha_corte_pagos', 'producto',
       'aplicativo', 'segmento', 'valor_cuota_mes', 'pago_total',
       'fecha_pago_minima', 'fecha_pago_maxima', 'porc_pago', 'marca_pago',
       'ajustes_banco', 'var_rpta_alt', 'tipo_var_rpta

In [85]:
resultado_oot = resultado_oot.rename(columns={
    'producto': 'producto_pagos',
    'aplicativo': 'aplicativo_pagos',
    'valor_cuota_mes': 'valor_cuota_mes_pagos',
    'marca_pago': 'marca_pago_pagos',
})

In [86]:
resultado_oot = resultado_oot[col_id + col_pred + col_vars]
resultado_oot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112549 entries, 0 to 112548
Data columns (total 37 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   nit_enmascarado             112549 non-null  int64  
 1   num_oblig_orig_enmascarado  112549 non-null  int64  
 2   num_oblig_enmascarado       112549 non-null  int64  
 3   fecha_analisis              112549 non-null  int32  
 4   fecha_var_rpta_alt          112549 non-null  int64  
 5   var_rpta_alt                0 non-null       Int64  
 6   tipo_var_rpta_alt           112549 non-null  object 
 7   prob_propension             112289 non-null  float64
 8   prob_alrt_temprana          112289 non-null  float64
 9   prob_auto_cura              112289 non-null  float64
 10  genero_cli                  88026 non-null   object 
 11  ano_nac_cli                 88032 non-null   float64
 12  edad_cli                    88032 non-null   float64
 13  estado_civil  

In [87]:
resultado["uso"]= "TRAINTEST"
resultado_oot["uso"]= "OOT"

In [88]:
coinciden = list(resultado.columns) == list(resultado_oot.columns)
print(f"¿Las columnas coinciden?: {coinciden}")
assert coinciden, "Las columnas no coinciden"

¿Las columnas coinciden?: True


In [89]:
base_final = pd.concat([resultado, resultado_oot], axis=0, ignore_index=True)
base_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 680800 entries, 0 to 680799
Data columns (total 38 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   nit_enmascarado             680800 non-null  int64  
 1   num_oblig_orig_enmascarado  680800 non-null  int64  
 2   num_oblig_enmascarado       680800 non-null  int64  
 3   fecha_analisis              680800 non-null  int32  
 4   fecha_var_rpta_alt          680800 non-null  int64  
 5   var_rpta_alt                568251 non-null  Int64  
 6   tipo_var_rpta_alt           680800 non-null  object 
 7   prob_propension             678652 non-null  float64
 8   prob_alrt_temprana          678652 non-null  float64
 9   prob_auto_cura              678652 non-null  float64
 10  genero_cli                  532515 non-null  object 
 11  ano_nac_cli                 532535 non-null  float64
 12  edad_cli                    532535 non-null  float64
 13  estado_civil  

In [90]:
categoricas = identificar_variables_categoricas(base_final, columnas=col_vars)
print(categoricas['resumen'].to_string(index=False))

            columna tipo_dato  n_valores_unicos  tiene_nulos
         genero_cli    object                 2         True
    cli_actualizado    object                 2         True
       estado_civil    object                 7         True
               segm    object                 7         True
   aplicativo_pagos    object                 7         True
   marca_pago_pagos    object                 7         True
          region_of    object                 8         True
      origen_fondos    object                10         True
canal_actualizacion    object                12         True
               ocup    object                14         True
            subsegm    object                16         True
     producto_pagos    object                16         True
   nombre_dpto_dirp    object                41         True


In [91]:
for col in categoricas['variables_categoricas']:
    valores = categoricas['valores_unicos'][col]
    print(f"\n{col} ({len(valores)} valores únicos):")
    print(valores)


genero_cli (2 valores únicos):
['F', 'M']

estado_civil (7 valores únicos):
['CASADO', 'DIVORCIADO', 'NO INFORMA', 'OTRO', 'SOLTERO', 'UNION LIBRE', 'VIUDO']

ocup (14 valores únicos):
['AGRICULTOR', 'AMA DE CASA', 'COMERCIANTE', 'DESEMPLEADO CON INGRESOS', 'DESEMPLEADO SIN INGRESOS', 'EMPLEADO', 'ESTUDIANTE', 'GANADERO', 'INDEPENDIENTE', 'OTRA', 'PENSIONADO', 'PROFESIONAL INDEPENDIENTE', 'RENTISTA DE CAPITAL', 'SOCIO O EMPLEADO - SOCIO']

origen_fondos (10 valores únicos):
['CAPITALIZACION POR PARTE DE LOS SOCIOS', 'DESARROLLO DEL OBJETO SOCIAL DE LA PJ', 'DIVIDENDOS Y PARTICIPACIONES', 'HONORARIOS Y COMISIONES', 'INTERESES Y RENDIMIENTOS FINANCIEROS', 'OTROS', 'RENDIMIENTOS POR INVERSIÓN', 'SALARIO Y DEMAS PAGOS LABORALES', 'UTILIDADES DEL NEGOCIO', 'VENTAS NETAS']

canal_actualizacion (12 valores únicos):
['ACD', 'ACO', 'ASS', 'AUD', 'BRO', 'CMI', 'DIG', 'GER', 'PIC', 'SDI', 'SVP', 'VTC']

cli_actualizado (2 valores únicos):
['N', 'S']

segm (7 valores únicos):
['EMPRESARIAL', 'IND

### 4.2.7 Conclusiones del análisis exploratorio de variables

#### 🔍 Aspectos relevantes identificados

**Calidad y Duplicidad de Datos:**
- Se identificaron datos duplicados en múltiples bases:
  - `base_info_modelos`: sin duplicados significativos a nivel cliente-obligación-fecha
  - `base_info_socio`: 1 registro duplicado (cliente 348225, fecha 20230831) que fue eliminado
  - `base_info_pagos`: 2 registros duplicados con variaciones en `valor_cuota_mes` y `porc_pago` que fueron eliminados aleatoriamente
- La deduplicación fue crítica para evitar sesgos en el cruce y asegurar correspondencia 1:1 en las claves de unión

**Sinergia de Información entre Fuentes:**
- Las variables de modelos (`alerta_temprana`, `auto_cura`, `propension_pago`) capturan comportamiento predictivo ya validado
- Las variables sociodemográficas complementan con contexto del cliente (rango etario, ingresos, estado laboral)
- Las variables de pagos aportan dinámicas recientes (frecuencia de pago, patrones de atraso)
- **Sinergia esperada**: la combinación de predicciones históricas + contexto demográfico + comportamiento reciente debería mejorar la discriminación entre clientes con riesgo de incumplimiento
- **Nota crítica**: la completitud de información varía por cliente; algunos pueden carecer de histórico en una o más fuentes, lo que se refleja en los NAs post-cruce

**Filtrado y Limpieza de Variables:**
- **Eliminación por nulos (>60%)**: Varias columnas descartadas por alta tasa de datos faltantes, indicativo de información incompleta en fuentes
- **Eliminación por concurrencia (>80% en valor dominante)**: Variables cuasi-constantes (ej. `cli_actualizado` casi siempre = 1) sin poder discriminativo
- **Resultado**: De X variables originales se retuvieron col_vars variables útiles para modelado
- El filtrado agresivo priorizó calidad sobre cantidad, facilitando la interpretabilidad del modelo

**Estructura de Variables Resultantes:**
- **Columnas de identificación (col_id)**: nit_enmascarado, num_oblig_orig_enmascarado, num_oblig_enmascarado, fecha_analisis
- **Columnas predicientes (col_pred)**: fecha_var_rpta_alt, var_rpta_alt, tipo_var_rpta_alt
- **Columnas predictoras (col_vars)**: Mezcla de variables numéricas y categóricas después de cruce
- **Columna de segmentación (uso)**: Etiqueta TRAINTEST vs OOT para separación de datasets

---

#### 🎯 Implicaciones para el Desempeño del Modelo

Basándose en la estructura y calidad del dataset post-limpieza, se anticipan los siguientes impactos:

**Fortalezas esperadas:**
- **Reducción de ruido**: el filtrado agresivo (>60% nulos, >80% constantes) minimiza variables no informativas que degradarían el modelo
- **Balance variable**: la retención de col_vars variables (mezcla numérica-categórica) permite capturar patrones lineales y no lineales
- **Información complementaria**: la integración de 4 fuentes garantiza que el modelo no dependa de una sola perspectiva del cliente

**Capacidad discriminativa esperada:**
- El modelo debería desempeñarse mejor en clientes con histórico completo en todas las fuentes (mayoría de la población TRAINTEST)
- Potencial desempeño degradado en OOT si contiene clientes nuevos o con pocos registros históricos
- Generalización moderada: robusto para el segmento "Personal" (dominante) pero puede ser menos preciso en otros segmentos

---


## 5. Separación base entrenamiento y validación

In [92]:
# Valores únicos de fecha_analisis para TRAINTEST ordenados de menor a mayor
fechas_unicas = sorted(base_final[(base_final["uso"]=="TRAINTEST")]["fecha_analisis"].unique())
print(f"Fechas únicas en TRAINTEST ({len(fechas_unicas)} valores):")
print(fechas_unicas)

Fechas únicas en TRAINTEST (5 valores):
[202307, 202308, 202309, 202310, 202311]


**Nota:** Para la selección de datos de entrenamiento y validacion se va a generar una separacion a nivel temporal donde para las fechas de analisis 202307, 202308, 202309, 202310 se van a emplear sus registros y 202311 se va a usar para validaciones. No obstante, dado que pueden existir clientes en las bases de entrenamiento, se va hacer un proceso de exclusion de estos clientes en la base de entrenamiento.

In [93]:
base_final['uso_detalle'] = base_final.apply(crear_uso_detalle, axis=1)
print("Distribución inicial de uso_detalle:")
print(base_final['uso_detalle'].value_counts(dropna=False))

Distribución inicial de uso_detalle:
uso_detalle
train    467785
val      112549
test     100466
Name: count, dtype: int64


In [94]:
nits_train = set(base_final[base_final['uso_detalle'] == 'train']['nit_enmascarado'].unique())
nits_test = set(base_final[base_final['uso_detalle'] == 'test']['nit_enmascarado'].unique())
nits_en_ambas = nits_train.intersection(nits_test)
ajustar_fn = crear_ajustar_uso_detalle(nits_en_ambas)

In [95]:
base_final['uso_detalle'] = base_final.apply(ajustar_fn, axis=1)

In [96]:
print("Distribución FINAL de uso_detalle:")
print(base_final['uso_detalle'].value_counts(dropna=False))

Distribución FINAL de uso_detalle:
uso_detalle
train            467785
val              112549
excluido_test     60276
test              40190
Name: count, dtype: int64


In [97]:
len(base_final)

680800

In [98]:
# Calcular proporción de variable respuesta por fecha
df_prop_fecha = calcular_porcentaje_por_fecha(base_final[base_final['uso_detalle']=="train"])
print("\n📊 Proporción de variable respuesta por fecha para la base de train:")
print(df_prop_fecha.to_string(index=False))


📊 Proporción de variable respuesta por fecha para la base de train:
 fecha_var_rpta_alt  var_rpta_alt  cantidad_casos  porcentaje
             202308             0           58590       51.61
             202308             1           54941       48.39
             202309             0           61357       50.63
             202309             1           59828       49.37
             202310             0           62236       53.69
             202310             1           53687       46.31
             202311             0           59875       51.11
             202311             1           57271       48.89


In [99]:
conteos = base_final[base_final['uso_detalle']=="train"]['segm'].value_counts()
total = conteos.sum()

print("\n" + "="*70)
print("DISTRIBUCIÓN DE CASOS POR SEGMENTO")
print("="*70)

for segmento, cantidad in conteos.items():
    porcentaje = (cantidad / total * 100)
    barra = "█" * int(porcentaje / 2)
    print(f"{segmento:20} {cantidad:8,} ({porcentaje:5.2f}%) {barra}")

print("="*70)
print(f"{'TOTAL':20} {total:8,} (100.00%)")
print("="*70)


DISTRIBUCIÓN DE CASOS POR SEGMENTO
PERSONAL              241,798 (64.36%) ████████████████████████████████
PLUS                   70,798 (18.84%) █████████
INDEPENDIENTES         38,119 (10.15%) █████
PYMES                  12,345 ( 3.29%) █
SOCIAL                 10,806 ( 2.88%) █
PREFERENCIAL            1,829 ( 0.49%) 
EMPRESARIAL                 3 ( 0.00%) 
TOTAL                 375,698 (100.00%)


In [100]:
num_colums =[col for col in col_vars if col not in identificar_variables_categoricas(base_final, columnas=col_vars)['variables_categoricas']]
base_final_limpia, resumen_limpieza = limpiar_datos_numericos(
    base_final,
    columnas=num_colums, 
    reemplazar_inf_por=0,
    reemplazar_nan_por=0,
    verbose=True
)
print("\nBase lista para entrenar:")
print(base_final_limpia.info())

REPORTE DE LIMPIEZA DE DATOS NUMÉRICOS

Columnas numéricas a procesar: 17
Columnas: ['prob_propension', 'prob_alrt_temprana', 'prob_auto_cura', 'ano_nac_cli', 'edad_cli', 'total_ing', 'tot_activos', 'tot_pasivos', 'f_vinc', 'f_ult_mantenimiento', 'egresos_mes', 'tot_patrimonio', 'valor_cuota_mes_pagos', 'pago_total', 'fecha_pago_minima', 'fecha_pago_maxima', 'porc_pago']

📊 ESTADO ANTES DE LA LIMPIEZA:
--------------------------------------------------------------------------------
  Total infinitos: 31
  Total NaNs: 2615477

  Infinitos por columna:
    porc_pago: 31

  NaNs por columna:
    valor_cuota_mes_pagos: 274825
    pago_total: 274825
    fecha_pago_minima: 274825
    fecha_pago_maxima: 274825
    porc_pago: 274825
    ano_nac_cli: 148265
    edad_cli: 148265
    total_ing: 134054
    tot_activos: 134054
    tot_pasivos: 134054
    f_vinc: 134054
    f_ult_mantenimiento: 134054
    egresos_mes: 134054
    tot_patrimonio: 134054
    prob_propension: 2148
    prob_alrt_temprana

In [101]:
print(col_id)

['nit_enmascarado', 'num_oblig_orig_enmascarado', 'num_oblig_enmascarado', 'fecha_analisis']


In [102]:
print(col_pred)

['fecha_var_rpta_alt', 'var_rpta_alt', 'tipo_var_rpta_alt']


In [103]:
print(identificar_variables_categoricas(base_final_limpia, columnas=col_vars)['variables_categoricas'])

['genero_cli', 'estado_civil', 'ocup', 'origen_fondos', 'canal_actualizacion', 'cli_actualizado', 'segm', 'subsegm', 'region_of', 'nombre_dpto_dirp', 'producto_pagos', 'aplicativo_pagos', 'marca_pago_pagos']


In [104]:
print([col for col in col_vars if col not in identificar_variables_categoricas(base_final_limpia, columnas=col_vars)['variables_categoricas']])

['prob_propension', 'prob_alrt_temprana', 'prob_auto_cura', 'ano_nac_cli', 'edad_cli', 'total_ing', 'tot_activos', 'tot_pasivos', 'f_vinc', 'f_ult_mantenimiento', 'egresos_mes', 'tot_patrimonio', 'valor_cuota_mes_pagos', 'pago_total', 'fecha_pago_minima', 'fecha_pago_maxima', 'porc_pago']


In [105]:
# base_final_limpia[base_final_limpia['uso_detalle']!="excluido_test"].to_csv(data_final / "base_final.csv")
base_final_limpia.to_csv(data_final / "base_final.csv")

In [106]:
# nombre_tabla="base_prueba"
# sp.subir_df(base_final_limpia[base_final_limpia['uso_detalle']!="excluido_test"], '{}.{}_{}'.format(zona_p,indice,nombre_tabla), modo='overwrite')
# sp.subir_df(base_final_limpia, '{}.{}_{}'.format(zona_p,indice,nombre_tabla), modo='overwrite')